In [ ]:
"""
STEP 02 - Build the analysis dataset (cohort) from the raw NHANES files.

Thesis: Risk Factor Identification and Predictive Modeling of Stroke using ML Techniques
Part 2 of the NHANES pipeline.

What it does (each block in the code is labelled STEP 2.x so you can track it):
  STEP 2.1  Load the raw files of each cycle and merge them into one row per person (by SEQN)
  STEP 2.2  Apply the inclusion rules and record how many people each rule removes (flow)
  STEP 2.3  Create clean, readable columns (target + features) from the raw NHANES codes
  STEP 2.4  Run sanity checks (duplicates, value ranges)
  STEP 2.5  Save the dataset, a codebook and the flow table; print a simple summary

IMPORTANT - no leakage here:
  Only fixed, row-by-row recoding is done (e.g. "1 = yes" -> 1, average of 3 BP readings,
  a published eGFR formula). Nothing is learned from the data: NO imputation, NO scaling,
  NO resampling. Those happen later, inside the training folds only.

How to run (same folder as step 01):
    python step02_build_cohort.py

Outputs
  data_processed/nhanes_stroke_cohort.csv   one row per person, clean columns
  out/step02_flow.csv                       inclusion flow (for the thesis flow diagram)
  out/step02_codebook.csv                   what every column means
  out/step02_missingness.csv                missing % per column
"""

from __future__ import annotations

import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
RAW_DIR = ROOT / "data_raw" / "nhanes"
PROC_DIR = ROOT / "data_processed"
OUT_DIR = ROOT / "out"

CYCLES = {"2007-2008": "E", "2009-2010": "F", "2011-2012": "G",
          "2013-2014": "H", "2015-2016": "I", "2017-2018": "J"}

# Same component/column list as step 01
COMPONENTS: dict[str, list[str]] = {
    "DEMO": ["SDDSRVYR", "RIDSTATR", "RIAGENDR", "RIDAGEYR", "RIDRETH1", "DMDEDUC2",
             "DMDMARTL", "INDFMPIR", "RIDEXPRG", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"],
    "MCQ": ["MCQ160F", "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ300A"],
    "BPQ": ["BPQ020"],
    "BIOPRO": ["LBXSGL", "LBXSCR", "LBXSUA"],
    "BMX": ["BMXBMI", "BMXWAIST"],
    "SMQ": ["SMQ020", "SMQ040"],
    "DIQ": ["DIQ010"],
    "GHB": ["LBXGH"],
    "TCHOL": ["LBXTC"],
    "HDL": ["LBDHDD"],
    "BPX": ["BPXSY1", "BPXSY2", "BPXSY3", "BPXDI1", "BPXDI2", "BPXDI3"],
    "ALB_CR": ["URXUMA", "URXUCR"],
    "DPQ": [f"DPQ0{i}0" for i in range(1, 10)],
    "SLQ": ["SLQ050"],
}

N_CYCLES = len(CYCLES)


def read_xpt(path: Path) -> pd.DataFrame:
    df = pd.read_sas(path, format="xport", encoding="latin-1")
    df.columns = [c.upper() for c in df.columns]
    # Known NHANES/SAS-transport quirk: a stored 0 is read back as a tiny number (5.4e-79).
    # Convert these back to exact 0, otherwise answers coded 0 (e.g. PHQ-9 items) are lost.
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num].abs() < 1e-10, 0.0)
    return df


def yes_no(s: pd.Series) -> pd.Series:
    """NHANES yes/no questions: 1 = yes -> 1, 2 = no -> 0, 7/9 (refused/don't know) -> missing."""
    return s.map({1: 1.0, 2: 0.0})


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def egfr_ckd_epi_2021(scr_mg_dl: pd.Series, age: pd.Series, female: pd.Series) -> pd.Series:
    """CKD-EPI 2021 creatinine equation (race-free), mL/min/1.73m2."""
    kappa = np.where(female == 1, 0.7, 0.9)
    alpha = np.where(female == 1, -0.241, -0.302)
    ratio = scr_mg_dl / kappa
    egfr = (142 * np.minimum(ratio, 1) ** alpha * np.maximum(ratio, 1) ** -1.200
            * 0.9938 ** age * np.where(female == 1, 1.012, 1.0))
    return pd.Series(egfr, index=scr_mg_dl.index).where(scr_mg_dl > 0)


def main() -> int:
    print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # =====================================================================================
    # STEP 2.1  Load and merge: one row per person per cycle
    # =====================================================================================
    print("\nSTEP 2.1  Loading and merging raw files ...")
    frames = []
    for cycle, sfx in CYCLES.items():
        base = None
        for comp, cols in COMPONENTS.items():
            path = RAW_DIR / cycle / f"{comp}_{sfx}.xpt"
            if not path.exists():
                print(f"  ERROR: missing {path}. Run step01 first.")
                return 1
            df = read_xpt(path)
            keep = ["SEQN"] + [c for c in cols if c in df.columns]
            df = df[keep]
            if df["SEQN"].duplicated().any():
                print(f"  ERROR: duplicate SEQN inside {path.name}")
                return 1
            # DEMO is the master list of participants; every other file is joined onto it
            base = df if base is None else base.merge(df, on="SEQN", how="left")
        base.insert(1, "cycle", cycle)
        frames.append(base)
        print(f"  {cycle}: {len(base):,} participants")
    raw = pd.concat(frames, ignore_index=True)
    print(f"  All cycles together: {len(raw):,} participants")

    # =====================================================================================
    # STEP 2.2  Inclusion rules (flow)
    # =====================================================================================
    print("\nSTEP 2.2  Applying inclusion rules ...")
    flow = []

    def record(label: str, d: pd.DataFrame) -> None:
        flow.append(dict(step=label, remaining=len(d), strokes=int((d.get("MCQ160F") == 1).sum())))

    d = raw.copy()
    record("All NHANES participants 2007-2018", d)
    d = d[d["RIDAGEYR"] >= 20]
    record("Aged 20 or older", d)
    d = d[d["RIDSTATR"] == 2]
    record("Examined in the mobile clinic (MEC)", d)
    d = d[d["MCQ160F"].isin([1, 2])]
    record("Answered the stroke question yes/no", d)
    d = d[d["RIDEXPRG"] != 1]        # NaN (not asked / men / older women) is kept
    record("Not pregnant at examination", d)
    d = d.reset_index(drop=True)

    flow_df = pd.DataFrame(flow)
    flow_df["removed"] = (-flow_df["remaining"].diff()).fillna(0).astype(int)
    flow_df.to_csv(OUT_DIR / "step02_flow.csv", index=False)

    # =====================================================================================
    # STEP 2.3  Create clean columns
    # =====================================================================================
    print("STEP 2.3  Creating clean columns ...")
    c = pd.DataFrame(index=d.index)

    # --- IDs and survey design (NOT used as model features) -------------------------------
    c["seqn"] = d["SEQN"].astype("int64")
    c["cycle"] = d["cycle"]
    c["weight_mec_6cyc"] = d["WTMEC2YR"] / N_CYCLES      # pooled weight for 6 cycles
    c["psu"] = d["SDMVPSU"]
    c["strata"] = d["SDMVSTRA"]

    # --- 2.3a Target -----------------------------------------------------------------------
    c["stroke"] = (d["MCQ160F"] == 1).astype(int)

    # --- 2.3b Demographics -----------------------------------------------------------------
    c["age"] = d["RIDAGEYR"]                                         # top-coded at 80
    c["female"] = (d["RIAGENDR"] == 2).astype(int)
    c["race"] = d["RIDRETH1"].map({1: "Mexican American", 2: "Other Hispanic",
                                   3: "Non-Hispanic White", 4: "Non-Hispanic Black",
                                   5: "Other/Multiracial"})
    c["education"] = d["DMDEDUC2"].map({1: "1 <9th grade", 2: "2 9-11th grade",
                                        3: "3 High school/GED", 4: "4 Some college/AA",
                                        5: "5 College graduate+"})
    c["income_ratio"] = d["INDFMPIR"]                                # 0-5, top-coded at 5
    c["marital"] = d["DMDMARTL"].map({1: "Married/partner", 6: "Married/partner",
                                      2: "Widowed/divorced/separated", 3: "Widowed/divorced/separated",
                                      4: "Widowed/divorced/separated", 5: "Never married"})
    # Kaggle-style 'ever_married' (core model only). Assumption: living with partner (6) -> 0.
    c["ever_married"] = d["DMDMARTL"].map({1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 0.0, 6: 0.0})

    # --- 2.3c Blood pressure ---------------------------------------------------------------
    c["hypertension"] = yes_no(d["BPQ020"])
    sbp = d[["BPXSY1", "BPXSY2", "BPXSY3"]].where(lambda x: x > 0)
    dbp = d[["BPXDI1", "BPXDI2", "BPXDI3"]].where(lambda x: x > 0)   # a 0 reading = not measurable
    c["sbp"] = sbp.mean(axis=1, skipna=True)
    c["dbp"] = dbp.mean(axis=1, skipna=True)

    # --- 2.3d Blood sugar ------------------------------------------------------------------
    c["glucose"] = d["LBXSGL"]                                       # mg/dL, serum
    c["hba1c"] = d["LBXGH"]                                          # %
    c["diabetes"] = d["DIQ010"].map({1: 1.0, 2: 0.0, 3: 0.0})        # borderline (3) -> 0

    # --- 2.3e Body size --------------------------------------------------------------------
    c["bmi"] = d["BMXBMI"]
    c["waist"] = d["BMXWAIST"]

    # --- 2.3f Cholesterol ------------------------------------------------------------------
    c["total_chol"] = d["LBXTC"]
    c["hdl"] = d["LBDHDD"]

    # --- 2.3g Kidney and other blood tests -------------------------------------------------
    c["egfr"] = egfr_ckd_epi_2021(d["LBXSCR"], d["RIDAGEYR"], c["female"])
    uacr = d["URXUMA"] / d["URXUCR"] * 100                           # mg/L / mg/dL*100 = mg/g
    c["uacr"] = uacr.where((d["URXUCR"] > 0) & d["URXUMA"].notna())
    c["uric_acid"] = d["LBXSUA"]

    # --- 2.3h Heart disease and family history ---------------------------------------------
    hd = d[["MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E"]]
    any_yes = (hd == 1).any(axis=1)
    any_no = (hd == 2).any(axis=1)
    c["heart_disease"] = np.where(any_yes, 1.0, np.where(any_no, 0.0, np.nan))
    c["family_hist_mi"] = yes_no(d["MCQ300A"])   # close relative had heart attack before 50

    # --- 2.3i Smoking ----------------------------------------------------------------------
    smk = pd.Series(np.nan, index=d.index, dtype="object")
    smk[d["SMQ020"] == 2] = "never"
    smk[(d["SMQ020"] == 1) & (d["SMQ040"] == 3)] = "former"
    smk[(d["SMQ020"] == 1) & (d["SMQ040"].isin([1, 2]))] = "current"
    c["smoking"] = smk

    # --- 2.3j Sensitivity-only variables (possible reverse causation) ---------------------
    phq = d[[f"DPQ0{i}0" for i in range(1, 10)]].where(lambda x: x.isin([0, 1, 2, 3]))
    c["phq9"] = phq.sum(axis=1, min_count=9)                          # needs all 9 items
    c["sleep_trouble"] = yes_no(d["SLQ050"])

    # =====================================================================================
    # STEP 2.4  Sanity checks
    # =====================================================================================
    print("STEP 2.4  Running sanity checks ...")
    problems = []
    if c.duplicated(subset=["seqn"]).any():
        problems.append("duplicate person IDs (seqn)")
    ranges = {"age": (20, 80), "bmi": (10, 90), "sbp": (60, 280), "dbp": (20, 160),
              "glucose": (20, 900), "hba1c": (3, 20), "total_chol": (50, 800), "hdl": (5, 250),
              "egfr": (1, 200), "uric_acid": (0.5, 20), "income_ratio": (0, 5), "waist": (40, 200)}
    range_report = []
    for col, (lo, hi) in ranges.items():
        v = c[col].dropna()
        out_of_range = int(((v < lo) | (v > hi)).sum())
        range_report.append((col, v.min(), v.max(), out_of_range, lo, hi))
        if out_of_range:
            problems.append(f"{col}: {out_of_range} values outside {lo}-{hi}")

    # =====================================================================================
    # STEP 2.5  Save outputs
    # =====================================================================================
    out_csv = PROC_DIR / "nhanes_stroke_cohort.csv"
    c.to_csv(out_csv, index=False)
    digest = sha256(out_csv)

    features_main = ["age", "female", "race", "education", "income_ratio", "marital",
                     "hypertension", "sbp", "dbp", "glucose", "hba1c", "diabetes",
                     "bmi", "waist", "total_chol", "hdl", "egfr", "uacr", "uric_acid",
                     "heart_disease", "family_hist_mi", "smoking"]
    features_core = ["age", "female", "hypertension", "heart_disease", "glucose", "bmi",
                     "smoking", "ever_married"]
    features_sens = ["phq9", "sleep_trouble"]

    codebook = [
        ("stroke", "target", "Ever told had a stroke (MCQ160F): 1 yes, 0 no"),
        ("age", "main+core", "Age in years (RIDAGEYR), top-coded 80"),
        ("female", "main+core", "1 female, 0 male (RIAGENDR)"),
        ("race", "main", "Race/ethnicity, 5 groups (RIDRETH1)"),
        ("education", "main", "Highest education, 5 levels (DMDEDUC2)"),
        ("income_ratio", "main", "Family income / poverty line, 0-5 (INDFMPIR)"),
        ("marital", "main", "Married/partner, widowed/divorced/separated, never married (DMDMARTL)"),
        ("ever_married", "core", "1 ever married, 0 never married or living with partner (DMDMARTL)"),
        ("hypertension", "main+core", "Ever told high blood pressure: 1 yes, 0 no (BPQ020)"),
        ("sbp", "main", "Systolic BP, mean of up to 3 readings, mmHg (BPXSY1-3)"),
        ("dbp", "main", "Diastolic BP, mean of up to 3 readings, zeros removed, mmHg (BPXDI1-3)"),
        ("glucose", "main+core", "Serum glucose, mg/dL (LBXSGL)"),
        ("hba1c", "main", "Glycated haemoglobin, % (LBXGH)"),
        ("diabetes", "main", "Ever told diabetes: 1 yes, 0 no/borderline (DIQ010)"),
        ("bmi", "main+core", "Body mass index, kg/m2 (BMXBMI)"),
        ("waist", "main", "Waist circumference, cm (BMXWAIST)"),
        ("total_chol", "main", "Total cholesterol, mg/dL (LBXTC)"),
        ("hdl", "main", "HDL cholesterol, mg/dL (LBDHDD)"),
        ("egfr", "main", "eGFR, CKD-EPI 2021, mL/min/1.73m2 (from LBXSCR, age, sex)"),
        ("uacr", "main", "Urine albumin/creatinine ratio, mg/g (URXUMA/URXUCR)"),
        ("uric_acid", "main", "Serum uric acid, mg/dL (LBXSUA)"),
        ("heart_disease", "main+core", "Any of heart failure, CHD, angina, heart attack (MCQ160B-E)"),
        ("family_hist_mi", "main", "Close relative had heart attack before 50 (MCQ300A)"),
        ("smoking", "main+core", "never / former / current (SMQ020, SMQ040)"),
        ("phq9", "sensitivity", "PHQ-9 depression score 0-27, all 9 items required (DPQ010-090)"),
        ("sleep_trouble", "sensitivity", "Ever told doctor trouble sleeping (SLQ050)"),
        ("seqn", "id", "NHANES person ID"),
        ("cycle", "design", "NHANES cycle"),
        ("weight_mec_6cyc", "design", "Exam weight / 6 (for weighted odds ratios only)"),
        ("psu", "design", "Sampling cluster (SDMVPSU)"),
        ("strata", "design", "Sampling stratum (SDMVSTRA)"),
    ]
    pd.DataFrame(codebook, columns=["column", "role", "meaning"]).to_csv(OUT_DIR / "step02_codebook.csv", index=False)

    miss = (c[features_main + ["ever_married"] + features_sens].isna().mean() * 100).round(1)
    miss.rename("missing_pct").to_csv(OUT_DIR / "step02_missingness.csv")

    # ------------------------------------------------------------------ simple summary
    line = "=" * 70
    print("\n" + line)
    print("STEP 02 SUMMARY  (copy everything from here down and paste it back)")
    print(line)

    print("\n[A] WHO IS INCLUDED (flow)")
    for _, r in flow_df.iterrows():
        rem = f"  (removed {r['removed']:,})" if r["removed"] else ""
        print(f"  {r['step']:<40} {r['remaining']:>7,} people{rem}")

    n, k = len(c), int(c["stroke"].sum())
    print(f"\n[B] FINAL DATASET")
    print(f"  People: {n:,}")
    print(f"  Stroke: {k:,} ({100 * k / n:.2f}%)")
    print(f"  No stroke: {n - k:,}")
    print(f"  If a model always says 'no stroke' -> accuracy {100 * (n - k) / n:.1f}%, strokes found 0")

    print("\n[C] STROKE CASES PER CYCLE")
    for cyc, g in c.groupby("cycle"):
        print(f"  {cyc}: {int(g['stroke'].sum()):>4} of {len(g):>5,}  ({100 * g['stroke'].mean():.2f}%)")

    print("\n[D] EACH FEATURE: missing values, and stroke vs no-stroke")
    print("    (numbers = average; for yes/no features = % with 'yes')")
    s1, s0 = c[c["stroke"] == 1], c[c["stroke"] == 0]
    for col in features_main + features_sens:
        tag = " [sensitivity only]" if col in features_sens else ""
        m = 100 * c[col].isna().mean()
        if not pd.api.types.is_numeric_dtype(c[col]):
            top = c[col].value_counts(normalize=True).head(3)
            parts = ", ".join(f"{lvl} {100 * p:.0f}%" for lvl, p in top.items())
            print(f"  {col:<15} missing {m:5.1f}%  | most common: {parts}{tag}")
        elif set(c[col].dropna().unique()) <= {0, 1}:
            print(f"  {col:<15} missing {m:5.1f}%  | yes: stroke {100 * s1[col].mean():5.1f}%  vs  "
                  f"no stroke {100 * s0[col].mean():5.1f}%{tag}")
        else:
            print(f"  {col:<15} missing {m:5.1f}%  | average: stroke {s1[col].mean():7.1f}  vs  "
                  f"no stroke {s0[col].mean():7.1f}{tag}")

    print("\n[E] SANITY CHECKS")
    for col, lo_v, hi_v, bad, lo, hi in range_report:
        status = "OK" if bad == 0 else f"{bad} outside expected {lo}-{hi}"
        print(f"  {col:<13} min {lo_v:8.1f}  max {hi_v:8.1f}  -> {status}")
    print(f"  Duplicate person IDs: {'none' if not c['seqn'].duplicated().any() else 'FOUND'}")
    print(f"  Overall: {'all checks passed' if not problems else 'CHECK: ' + '; '.join(problems)}")

    print("\n[F] SAVED")
    print(f"  {out_csv.relative_to(ROOT)}  ({n:,} rows x {c.shape[1]} columns)")
    print(f"  sha256: {digest[:16]}")
    print("  out/step02_flow.csv, out/step02_codebook.csv, out/step02_missingness.csv")
    print(line)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [1]:
# ============================================================
# STEP 3.1  SETTINGS + SHARED FUNCTIONS  (run this cell first, every time you open the notebook)
# ============================================================
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
COHORT_CSV = ROOT / "data_processed" / "nhanes_stroke_cohort.csv"
COHORT_SHA16 = "7d209c0936fee395"            # fingerprint from Step 02
SPLIT_DIR = ROOT / "data_split"; SPLIT_DIR.mkdir(exist_ok=True)
OUT_DIR = ROOT / "out"; OUT_DIR.mkdir(exist_ok=True)
TRAIN_CSV, TEST_CSV = SPLIT_DIR / "train.csv", SPLIT_DIR / "test.csv"
SEED, TEST_SIZE, TARGET = 42, 0.20, "stroke"

# --- Feature lists ---
NUMERIC_MAIN = ["age", "income_ratio", "sbp", "dbp", "glucose", "hba1c", "bmi", "waist",
                "total_chol", "hdl", "egfr", "uacr", "uric_acid"]
BINARY_MAIN = ["female", "hypertension", "diabetes", "heart_disease", "family_hist_mi"]
CATEGORICAL_MAIN = ["race", "education", "marital", "smoking"]
FEATURES_MAIN = NUMERIC_MAIN + BINARY_MAIN + CATEGORICAL_MAIN          # 22 features (main model)

NUMERIC_CORE = ["age", "glucose", "bmi"]
BINARY_CORE = ["female", "hypertension", "heart_disease", "ever_married"]
CATEGORICAL_CORE = ["smoking"]
FEATURES_CORE = NUMERIC_CORE + BINARY_CORE + CATEGORICAL_CORE          # 8 features (Kaggle check)

SENSITIVITY_EXTRA = ["phq9", "sleep_trouble"]                          # sensitivity analysis only
LOG_FEATURES = ["uacr"]
DBP_MIN_VALID = 20      # fixed rule: diastolic BP < 20 = measurement error -> missing

def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

def fixed_cleaning(X):
    """Row-by-row fixed rules. Nothing is learned from data -> cannot leak."""
    X = X.copy()
    if "dbp" in X.columns:
        X.loc[X["dbp"] < DBP_MIN_VALID, "dbp"] = np.nan
    for col in LOG_FEATURES:
        if col in X.columns:
            X[col] = np.log1p(X[col].clip(lower=0))
    return X

def build_preprocessor(numeric, binary, categorical, scale=True):
    """Numbers: median + 'was missing' flag (+ scaling). Yes/No: most frequent + flag.
    Categories: 'missing' group + one-hot. Everything is learned in fit() = TRAIN only."""
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    ct = ColumnTransformer([
        ("num", Pipeline(num_steps), numeric),
        ("bin", SimpleImputer(strategy="most_frequent", add_indicator=True), binary),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         categorical),
    ], remainder="drop")
    return Pipeline([("clean", FunctionTransformer(fixed_cleaning, feature_names_out="one-to-one")),
                     ("columns", ct)])

print("STEP 3.1 done: settings and functions loaded.")

STEP 3.1 done: settings and functions loaded.


In [2]:
# ============================================================
# STEP 3.2  LOAD COHORT + LOCKED TRAIN/TEST SPLIT (80/20)
# ============================================================
assert sha16(COHORT_CSV) == COHORT_SHA16, "Cohort file changed! Re-run Step 02."
df = pd.read_csv(COHORT_CSV)

if TRAIN_CSV.exists() and TEST_CSV.exists():          # never re-draw the split
    train, test = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    print("Split already exists -> re-used.")
else:
    strat = df[TARGET].astype(str) + "_" + df["cycle"].astype(str)   # keep stroke rate AND year mix
    train, test = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=strat)
    train = train.sort_values("seqn").reset_index(drop=True)
    test = test.sort_values("seqn").reset_index(drop=True)
    train.to_csv(TRAIN_CSV, index=False)
    test.to_csv(TEST_CSV, index=False)
    print("New split created and saved.")

print("\n=== STEP 3.2 RESULT ===")
for name, part in [("TRAIN", train), ("TEST", test)]:
    k = int(part[TARGET].sum())
    print(f"{name:<5}: {len(part):>6,} people | {k:>5,} strokes ({100*k/len(part):.2f}%)")
print(f"TRAIN fingerprint: {sha16(TRAIN_CSV)}")
print(f"TEST  fingerprint: {sha16(TEST_CSV)}")
print("\nStroke rate per cycle (should be similar):")
for cyc in sorted(df["cycle"].unique()):
    a, b = train.loc[train.cycle == cyc, TARGET], test.loc[test.cycle == cyc, TARGET]
    print(f"  {cyc}:  TRAIN {100*a.mean():5.2f}%   TEST {100*b.mean():5.2f}%")
print("\nTEST is now LOCKED - only used once, at the very end.")

Split already exists -> re-used.

=== STEP 3.2 RESULT ===
TRAIN: 26,394 people | 1,057 strokes (4.00%)
TEST :  6,599 people |   265 strokes (4.02%)
TRAIN fingerprint: ec8ea15e1e9a4286
TEST  fingerprint: 8a5d289373fe4a35

Stroke rate per cycle (should be similar):
  2007-2008:  TRAIN  4.28%   TEST  4.26%
  2009-2010:  TRAIN  3.63%   TEST  3.67%
  2011-2012:  TRAIN  4.11%   TEST  4.09%
  2013-2014:  TRAIN  3.51%   TEST  3.53%
  2015-2016:  TRAIN  3.66%   TEST  3.70%
  2017-2018:  TRAIN  4.90%   TEST  4.90%

TEST is now LOCKED - only used once, at the very end.


In [ ]:
# ============================================================
# STEP 3.3  FIT PREPROCESSING ON TRAIN ONLY + LEAKAGE CHECKS
# ============================================================
prep = build_preprocessor(NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
prep.set_output(transform="pandas")
Xtr = prep.fit_transform(train[FEATURES_MAIN])      # learn from TRAIN
Xte = prep.transform(test[FEATURES_MAIN])           # only apply to TEST

prep_core = build_preprocessor(NUMERIC_CORE, BINARY_CORE, CATEGORICAL_CORE, scale=True)
prep_core.set_output(transform="pandas")
Xtr_core = prep_core.fit_transform(train[FEATURES_CORE])

learned = dict(zip(NUMERIC_MAIN, prep.named_steps["columns"].named_transformers_["num"]
                   .named_steps["impute"].statistics_))
train_medians = fixed_cleaning(train[NUMERIC_MAIN]).median()

checks = [
    ("No person in both TRAIN and TEST", len(set(train.seqn) & set(test.seqn)) == 0),
    ("TRAIN + TEST = whole cohort", len(train) + len(test) == len(df)),
    ("Fill-in medians learned from TRAIN only", all(np.isclose(learned[c], train_medians[c]) for c in NUMERIC_MAIN)),
    ("No missing values left (TRAIN)", not Xtr.isna().any().any()),
    ("No missing values left (TEST)", not Xte.isna().any().any()),
    ("TEST has same columns as TRAIN", list(Xte.columns) == list(Xtr.columns)),
    ("Scaling learned from TRAIN (TRAIN mean = 0)", bool(np.allclose(Xtr.iloc[:, :len(NUMERIC_MAIN)].mean(), 0, atol=1e-8))),
]

print("=== STEP 3.3 RESULT ===")
print(f"Fixed rule: diastolic BP < {DBP_MIN_VALID} -> missing ({int((df.dbp < DBP_MIN_VALID).sum())} values)")
print("Fixed rule: urine albumin/creatinine ratio -> log scale")
print("\nFill-in values learned from TRAIN (medians):")
for c in NUMERIC_MAIN:
    print(f"  {c:<13} {learned[c]:8.2f}" + ("  (log scale)" if c in LOG_FEATURES else ""))
flags = [c for c in Xtr.columns if "missingindicator" in c]
onehots = [c for c in Xtr.columns if c.startswith("cat__")]
print(f"\nMain model: 22 features -> {Xtr.shape[1]} input columns "
      f"(numbers {len(NUMERIC_MAIN)}, yes/no {len(BINARY_MAIN)}, category columns {len(onehots)}, missing-flags {len(flags)})")
print(f"Core model: 8 features -> {Xtr_core.shape[1]} input columns")
print("\nLeakage checks:")
for name, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
print("\nOverall:", "ALL PASSED" if all(ok for _, ok in checks) else "SOME FAILED - stop and tell Claude")

In [ ]:
 baseline lr · PY
# =====================================================================================
# STEP 04 - Baseline ("always no stroke") + Logistic Regression with NESTED cross-validation
# Thesis: Risk Factor Identification and Predictive Modeling of Stroke using ML Techniques
#
# HOW TO USE: copy each block (they are separated by cell markers) into its own notebook cell, in order.
# Needs: Cell 3.1 (settings + functions) already run in this notebook session.
#
# Everything here uses the TRAIN part only. The TEST part stays locked.
# Cell 4.1 is the "evaluation engine" re-used by every later model -> keep it in the notebook
# and re-run it (after Cell 3.1) whenever you restart the kernel.
# =====================================================================================
 
 
# %%
# ============================================================
# STEP 4.0  CHECK SETUP + LOAD TRAIN
# ============================================================
for name in ["build_preprocessor", "fixed_cleaning", "NUMERIC_MAIN", "TRAIN_CSV", "sha16"]:
    assert name in globals(), f"'{name}' not found -> run Cell 3.1 first"
assert sha16(TRAIN_CSV) == "ec8ea15e1e9a4286", "TRAIN file changed! Stop and tell Claude."
 
import time, warnings
import sklearn, imblearn
from scipy.optimize import brentq
from scipy.special import expit, logit
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.impute import MissingIndicator
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import FeatureUnion
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn import FunctionSampler
 
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
 
train = pd.read_csv(TRAIN_CSV)
X_train, y_train = train[FEATURES_MAIN], train[TARGET].values
print(f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__}")
print(f"TRAIN loaded: {len(train):,} people, {int(y_train.sum()):,} strokes ({100*y_train.mean():.2f}%)")

In [2]:
# ============================================================
# STEP 4.0  CHECK SETUP + LOAD TRAIN DATA
# (needs Cell 3.1 to be run first in this notebook session)
# ============================================================
for name in ["build_preprocessor", "fixed_cleaning", "NUMERIC_MAIN", "TRAIN_CSV", "sha16"]:
    assert name in globals(), f"'{name}' not found -> run Cell 3.1 first"
assert sha16(TRAIN_CSV) == "ec8ea15e1e9a4286", "TRAIN file changed! Stop and tell Claude."

import time, warnings
import sklearn, imblearn
from scipy.optimize import brentq
from scipy.special import expit, logit
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.impute import MissingIndicator
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_predict)
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.preprocessing import OrdinalEncoder
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn import FunctionSampler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

train = pd.read_csv(TRAIN_CSV)
X_train, y_train = train[FEATURES_MAIN], train[TARGET].values

print("=== STEP 4.0 RESULT ===")
print(f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__}")
print(f"TRAIN loaded: {len(train):,} people, {int(y_train.sum()):,} strokes ({100*y_train.mean():.2f}%)")
print("TEST is NOT loaded (stays locked).")

=== STEP 4.0 RESULT ===
scikit-learn 1.7.2 | imbalanced-learn 0.14.0
TRAIN loaded: 26,394 people, 1,057 strokes (4.00%)
TEST is NOT loaded (stays locked).


In [3]:
# ============================================================
# STEP 4.1  EVALUATION ENGINE  (re-used for every model in later steps)
# ============================================================
# --- Nested CV settings (fixed in the analysis plan) ---
OUTER_SPLITS, OUTER_REPEATS = 5, 3       # 15 outer folds -> honest performance estimate
INNER_SPLITS = 3                         # inner folds -> tuning, threshold, calibration
N_ITER = 30                              # random-search candidates per outer fold
TUNING_METRIC = "average_precision"      # PR-AUC (right metric for rare outcomes)
OUTER_CV = RepeatedStratifiedKFold(n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=SEED)
N_JOBS = -1                              # use all CPU cores


def _smotenc_after_numeric(X, y, n_num, seed=SEED):
    """SMOTE for mixed data: every column after the numeric block is categorical/binary."""
    cat_idx = list(range(n_num, X.shape[1]))
    return SMOTENC(categorical_features=cat_idx, random_state=seed).fit_resample(X, y)


def build_model_pipeline(estimator, numeric, binary, categorical, scale=True):
    """
    The SAME preprocessing rules as Step 3, arranged so SMOTE (if chosen) runs in the middle:
      1 fixed rules (dbp<20 -> missing, log UACR)
      2 fill missing (numbers: TRAIN-fold median | yes/no: most frequent | categories: 'missing')
        + 'was missing' flags
      3 optional SMOTENC (only on the training fold, never on validation/test rows)
      4 one-hot encode categories, scale numbers
      5 the model
    Everything is fitted inside each training fold -> no leakage.
    """
    n_num, n_bin, n_cat = len(numeric), len(binary), len(categorical)
    fill = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric),
        ("bin", SimpleImputer(strategy="most_frequent"), binary),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
         categorical),
        ("flags", MissingIndicator(features="missing-only"), numeric + binary),
    ], remainder="drop")
    encode = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", list(range(n_num))),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         list(range(n_num + n_bin, n_num + n_bin + n_cat))),
    ], remainder="passthrough")
    return ImbPipeline([
        ("clean", FunctionTransformer(fixed_cleaning)),
        ("fill", fill),
        ("sampler", "passthrough"),
        ("encode", encode),
        ("model", estimator),
    ])


def smote_sampler(n_num):
    return FunctionSampler(func=_smotenc_after_numeric, kw_args={"n_num": n_num}, validate=False)


def _calibration_slope_intercept(y, p):
    lp = logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp, y).coef_[0, 0]
    f = lambda a: np.sum(y - expit(a + lp.ravel()))          # intercept with slope fixed at 1
    try:
        intercept = brentq(f, -10, 10)
    except ValueError:
        intercept = np.nan
    return slope, intercept


def _thresholds_from_oof(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    t_youden = thr[np.argmax(tpr - fpr)]
    prec, rec, thr_pr = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    f2 = 5 * prec * rec / np.clip(4 * prec + rec, 1e-12, None)
    return {"youden": float(min(t_youden, 1.0)), "f1": float(thr_pr[np.argmax(f1)]),
            "f2": float(thr_pr[np.argmax(f2)])}


def _metrics_at(y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    rec = tp / (tp + fn) if tp + fn else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    return dict(recall=rec, specificity=tn / (tn + fp) if tn + fp else 0.0, precision=prec,
                npv=tn / (tn + fn) if tn + fn else 0.0, accuracy=(tp + tn) / len(y),
                f1=2 * prec * rec / (prec + rec) if prec + rec else 0.0,
                f2=5 * prec * rec / (4 * prec + rec) if prec + rec else 0.0,
                tp=int(tp), fn=int(fn), fp=int(fp), tn=int(tn))


def run_nested_cv(model_name, pipeline, param_dist, X, y, n_iter=N_ITER, verbose=True):
    """
    For each of the 15 outer folds:
      a) tune hyper-parameters on the outer-TRAIN part with an inner 3-fold random search
      b) on the outer-TRAIN part only: get out-of-fold predictions of the best setting ->
         learn Platt calibration + choose thresholds (Youden, F1, F2)
      c) score the untouched outer-VALIDATION part
    Returns per-fold results (DataFrame) and outer-validation predictions.
    """
    rows, preds = [], []
    t0 = time.time()
    for k, (tr, va) in enumerate(OUTER_CV.split(X, y), start=1):
        Xtr, Xva, ytr, yva = X.iloc[tr], X.iloc[va], y[tr], y[va]
        inner = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED + k)
        search = RandomizedSearchCV(pipeline, param_dist, n_iter=n_iter, scoring=TUNING_METRIC,
                                    cv=inner, random_state=SEED + k, n_jobs=N_JOBS, refit=True,
                                    error_score="raise")
        search.fit(Xtr, ytr)
        # (b) calibration + thresholds from inner out-of-fold predictions (outer-TRAIN only)
        oof = cross_val_predict(clone(search.best_estimator_), Xtr, ytr, cv=inner,
                                method="predict_proba", n_jobs=N_JOBS)[:, 1]
        platt = LogisticRegression(C=1e6, max_iter=1000).fit(
            logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1), ytr)
        cal = lambda p: platt.predict_proba(logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]
        thr = _thresholds_from_oof(ytr, cal(oof))
        # (c) score the untouched outer-validation part
        p_va = cal(search.best_estimator_.predict_proba(Xva)[:, 1])
        slope, intercept = _calibration_slope_intercept(yva, p_va)
        row = dict(model=model_name, fold=k, roc_auc=roc_auc_score(yva, p_va),
                   pr_auc=average_precision_score(yva, p_va), brier=brier_score_loss(yva, p_va),
                   cal_slope=slope, cal_intercept=intercept,
                   thr_youden=thr["youden"], thr_f1=thr["f1"], thr_f2=thr["f2"],
                   best_params=str(search.best_params_))
        for tname in ["youden", "f2", "f1"]:
            for m, v in _metrics_at(yva, p_va, thr[tname]).items():
                row[f"{tname}_{m}"] = v
        rows.append(row)
        preds.append(pd.DataFrame({"model": model_name, "fold": k, "row": va, "y": yva, "p": p_va}))
        if verbose:
            print(f"  fold {k:>2}/15  PR-AUC {row['pr_auc']:.3f}  ROC-AUC {row['roc_auc']:.3f}  "
                  f"recall {row['youden_recall']:.2f}  accuracy {row['youden_accuracy']:.2f}  "
                  f"({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows), pd.concat(preds, ignore_index=True)


def summarise(res, title):
    """Simple, row-wise summary: mean (SD) over the 15 outer folds."""
    def ms(col, pct=False):
        v = res[col] * (100 if pct else 1)
        return f"{v.mean():.1f}% (±{v.std():.1f})" if pct else f"{v.mean():.3f} (±{v.std():.3f})"
    print(f"\n--- {title}: average over 15 outer folds (± = SD) ---")
    print(f"  ROC-AUC                {ms('roc_auc')}")
    print(f"  PR-AUC                 {ms('pr_auc')}   (no-skill = {100*y_train.mean():.1f}%)")
    print(f"  Brier score            {ms('brier')}   (lower is better)")
    print(f"  Calibration slope      {ms('cal_slope')}   (ideal 1.0)")
    print(f"  Calibration intercept  {ms('cal_intercept')}   (ideal 0.0)")
    print("  At the main threshold (Youden):")
    print(f"    Recall (strokes found)        {ms('youden_recall', True)}")
    print(f"    Accuracy                      {ms('youden_accuracy', True)}")
    print(f"    Specificity                   {ms('youden_specificity', True)}")
    print(f"    Precision                     {ms('youden_precision', True)}")
    print(f"    NPV                           {ms('youden_npv', True)}")
    print(f"    F1 / F2                       {res['youden_f1'].mean():.3f} / {res['youden_f2'].mean():.3f}")
    print(f"  Alternative threshold (max-F2): recall {ms('f2_recall', True)}, accuracy {ms('f2_accuracy', True)}")

print("STEP 4.1 done: evaluation engine loaded "
      f"({OUTER_SPLITS}x{OUTER_REPEATS} outer folds, {INNER_SPLITS} inner folds, {N_ITER} candidates).")

STEP 4.1 done: evaluation engine loaded (5x3 outer folds, 3 inner folds, 30 candidates).


In [ ]:

# ============================================================
# STEP 4.2  BASELINE: a "model" that always says "no stroke"  (the accuracy paradox)
# ============================================================
base_rows = []
for k, (tr, va) in enumerate(OUTER_CV.split(X_train, y_train), start=1):
    m = _metrics_at(y_train[va], np.zeros(len(va)), 0.5)
    base_rows.append(dict(model="Always no stroke", fold=k, **m))
baseline = pd.DataFrame(base_rows)
baseline.to_csv(OUT_DIR / "step04_baseline_folds.csv", index=False)
 
print("=== STEP 4.2 RESULT: BASELINE ('always no stroke') ===")
print(f"  Accuracy         {100*baseline['accuracy'].mean():.1f}%   <- looks excellent ...")
print(f"  Recall           {100*baseline['recall'].mean():.1f}%    <- ... but it finds ZERO strokes")
print(f"  Strokes missed   {baseline['fn'].sum() // OUTER_REPEATS:,} of {int(y_train.sum()):,} "
      "(every TRAIN stroke is checked once per repeat)")
print("  This is the ACCURACY PARADOX. Any model must beat this on RECALL, not accuracy.")

In [ ]:
# ============================================================
# STEP 4.3  LOGISTIC REGRESSION - nested CV  (takes a few minutes)
# ============================================================
lr_pipe = build_model_pipeline(LogisticRegression(solver="liblinear", max_iter=5000),
                               NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
lr_params = {
    "model__C": loguniform(1e-3, 1e2),                      # strength of regularisation
    "model__penalty": ["l1", "l2"],                         # lasso or ridge
    "model__class_weight": [None, "balanced"],              # imbalance option A
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option B (SMOTENC)
}
print("=== STEP 4.3 RUNNING: Logistic Regression (15 outer folds) ===")
lr_res, lr_pred = run_nested_cv("Logistic Regression", lr_pipe, lr_params, X_train, y_train)
lr_res.to_csv(OUT_DIR / "step04_LR_folds.csv", index=False)
lr_pred.to_csv(OUT_DIR / "step04_LR_outer_predictions.csv", index=False)

In [ ]:
# ============================================================
# STEP 4.4  SUMMARY (paste this output back to Claude)
# ============================================================
print("=" * 66)
print("STEP 04 SUMMARY")
print("=" * 66)
print("\n[A] BASELINE - always says 'no stroke'")
print(f"  Accuracy {100*baseline['accuracy'].mean():.1f}%  |  Recall {100*baseline['recall'].mean():.1f}%  |  strokes found 0")

summarise(lr_res, "[B] LOGISTIC REGRESSION")

found = lr_res["youden_tp"].sum() / OUTER_REPEATS          # each person is validated once per repeat
total = int(y_train.sum())
print(f"\n[C] IN PLAIN WORDS (each TRAIN person is checked once per repeat; average of 3 repeats)")
print(f"  Baseline found 0 of {total:,} strokes.")
print(f"  Logistic Regression found about {found:,.0f} of {total:,} strokes ({100*found/total:.1f}%),")
print(f"  while its accuracy was {100*lr_res['youden_accuracy'].mean():.1f}% (lower than the baseline - and that is fine).")

print("\n[D] SETTINGS CHOSEN BY THE INNER TUNING (how often, out of 15 folds)")
bp = lr_res["best_params"]
n_smote = bp.str.contains("FunctionSampler").sum()
n_bal = bp.str.contains("'balanced'").sum()
n_l1 = bp.str.contains("'l1'").sum()
print(f"  SMOTE used            {n_smote} / 15")
print(f"  class_weight=balanced {n_bal} / 15")
print(f"  L1 (lasso) penalty    {n_l1} / 15")
print("\n[E] SAVED: out/step04_baseline_folds.csv, out/step04_LR_folds.csv, out/step04_LR_outer_predictions.csv")
print("=" * 66)

In [ ]:
# ============================================================
# STEP 5.1  LOGISTIC REGRESSION WITH SPLINES - nested CV
# Same engine, same 15 folds, same tuning budget as Step 4.3.
# Splines let LR model curved effects (e.g. risk rising faster at older ages).
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from sklearn.preprocessing import SplineTransformer

MODEL_NAME = "LR + splines"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 20-35 min). "
      "A line appears after each of the 15 folds.", flush=True)

spl_pipe = build_model_pipeline(LogisticRegression(solver="liblinear", max_iter=5000),
                                NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
# numbers: scale -> spline basis (curved effects); categories/yes-no unchanged
spl_pipe.set_params(encode__num=Pipeline([("scale", StandardScaler()),
                                          ("spline", SplineTransformer(degree=3, n_knots=4))]))
spl_params = {
    "encode__num__spline__n_knots": [3, 4, 5],              # flexibility of the curves
    "model__C": loguniform(1e-3, 1e2),
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"],
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
spl_res, spl_pred = run_nested_cv(MODEL_NAME, spl_pipe, spl_params, X_train, y_train)
spl_res.to_csv(OUT_DIR / "step05_LR_splines_folds.csv", index=False)
spl_pred.to_csv(OUT_DIR / "step05_LR_splines_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.1 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(spl_res, MODEL_NAME)
bp = spl_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced {bp.str.contains(chr(39) + 'balanced' + chr(39)).sum()}, "
      f"L1 {bp.str.contains(chr(39) + 'l1' + chr(39)).sum()}")
print("Saved: out/step05_LR_splines_folds.csv")
print("=" * 66)

In [ ]:
# ============================================================
# STEP 5.3  DECISION TREE - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# A single tree = a flowchart of yes/no questions. Trees do not need scaling.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from scipy.stats import randint
from sklearn.tree import DecisionTreeClassifier

MODEL_NAME = "Decision Tree"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 15-30 min). "
      "A line appears after each of the 15 folds.", flush=True)

dt_pipe = build_model_pipeline(DecisionTreeClassifier(random_state=SEED),
                               NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
dt_params = {
    "model__max_depth": randint(2, 13),                   # how many questions deep
    "model__min_samples_leaf": randint(5, 201),           # smallest group at the end of a branch
    "model__criterion": ["gini", "entropy"],
    "model__ccp_alpha": loguniform(1e-5, 1e-2),           # pruning (removes weak branches)
    "model__class_weight": [None, "balanced"],
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
dt_res, dt_pred = run_nested_cv(MODEL_NAME, dt_pipe, dt_params, X_train, y_train)
dt_res.to_csv(OUT_DIR / "step05_DT_folds.csv", index=False)
dt_pred.to_csv(OUT_DIR / "step05_DT_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.3 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(dt_res, MODEL_NAME)
bp = dt_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced {bp.str.contains(chr(39) + 'balanced' + chr(39)).sum()}")
print("Saved: out/step05_DT_folds.csv")
print("=" * 66)

In [ ]:
# ============================================================
# STEP 5.4  RANDOM FOREST - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# Many decision trees, each trained on a random sample; they vote.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier

MODEL_NAME = "Random Forest"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 60-120 min - consider running it "
      "with the laptop plugged in and 'caffeinate -i' in Terminal). "
      "A line appears after each of the 15 folds.", flush=True)

rf_pipe = build_model_pipeline(RandomForestClassifier(random_state=SEED, n_jobs=1),
                               NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
rf_params = {
    "model__n_estimators": randint(200, 501),             # number of trees
    "model__max_depth": [None, 4, 6, 8, 12, 16],          # how deep each tree can grow
    "model__min_samples_leaf": randint(1, 51),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],    # features tried at each split
    "model__class_weight": [None, "balanced", "balanced_subsample"],
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
rf_res, rf_pred = run_nested_cv(MODEL_NAME, rf_pipe, rf_params, X_train, y_train)
rf_res.to_csv(OUT_DIR / "step05_RF_folds.csv", index=False)
rf_pred.to_csv(OUT_DIR / "step05_RF_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.4 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(rf_res, MODEL_NAME)
bp = rf_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced/balanced_subsample {bp.str.contains('balanced').sum()}")
print("Saved: out/step05_RF_folds.csv")
print("=" * 66)

In [ ]:
# ============================================================
# STEP 5.5  EXTRA TREES - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# Like Random Forest, but split points are chosen at random -> faster, often smoother.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from scipy.stats import randint
from sklearn.ensemble import ExtraTreesClassifier

MODEL_NAME = "Extra Trees"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 45-100 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

et_pipe = build_model_pipeline(ExtraTreesClassifier(random_state=SEED, n_jobs=1),
                               NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
et_params = {
    "model__n_estimators": randint(200, 501),
    "model__max_depth": [None, 4, 6, 8, 12, 16],
    "model__min_samples_leaf": randint(1, 51),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
et_res, et_pred = run_nested_cv(MODEL_NAME, et_pipe, et_params, X_train, y_train)
et_res.to_csv(OUT_DIR / "step05_ET_folds.csv", index=False)
et_pred.to_csv(OUT_DIR / "step05_ET_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.5 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(et_res, MODEL_NAME)
bp = et_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced/balanced_subsample {bp.str.contains('balanced').sum()}")
print("Saved: out/step05_ET_folds.csv")
print("=" * 66)

In [ ]:
# ============================================================
# STEP 5.6  BALANCED RANDOM FOREST - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# A Random Forest built for rare outcomes: every tree is trained on ALL strokes plus
# an equal-sized random sample of non-strokes (inside the training fold only).
# Its raw probabilities are inflated by design -> the engine's Platt calibration corrects them.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from scipy.stats import randint
from imblearn.ensemble import BalancedRandomForestClassifier

MODEL_NAME = "Balanced Random Forest"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 30-70 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

brf_pipe = build_model_pipeline(
    BalancedRandomForestClassifier(random_state=SEED, n_jobs=1,
                                   sampling_strategy="auto", replacement=False, bootstrap=True),
    NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
brf_params = {
    "model__n_estimators": randint(200, 501),
    "model__max_depth": [None, 4, 6, 8, 12, 16],
    "model__min_samples_leaf": randint(1, 51),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
brf_res, brf_pred = run_nested_cv(MODEL_NAME, brf_pipe, brf_params, X_train, y_train)
brf_res.to_csv(OUT_DIR / "step05_BRF_folds.csv", index=False)
brf_pred.to_csv(OUT_DIR / "step05_BRF_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.6 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(brf_res, MODEL_NAME)
bp = brf_res["best_params"]
print(f"\nSettings chosen (out of 15 folds): SMOTE {bp.str.contains('FunctionSampler').sum()}")
print("Saved: out/step05_BRF_folds.csv")
print("=" * 66)

In [4]:
# ============================================================
# STEP 5.7  XGBOOST - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# Boosting: trees are built one after another, each fixing the mistakes of the previous ones.
# Imbalance option A for XGBoost = scale_pos_weight (its version of class_weight="balanced").
# Needs: Cells 3.1, 4.0, 4.1 run in this session. Needs the 'xgboost' package.
# ============================================================
from scipy.stats import randint, uniform
import xgboost
from xgboost import XGBClassifier

MODEL_NAME = "XGBoost"
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())   # about 24 non-strokes per stroke
print(f"xgboost {xgboost.__version__}")
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 40-90 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

xgb_pipe = build_model_pipeline(
    XGBClassifier(tree_method="hist", eval_metric="logloss", random_state=SEED, n_jobs=1, verbosity=0),
    NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
xgb_params = {
    "model__n_estimators": randint(100, 601),            # number of boosting rounds
    "model__learning_rate": loguniform(0.01, 0.3),       # step size
    "model__max_depth": randint(2, 9),                   # depth of each tree
    "model__min_child_weight": randint(1, 21),
    "model__subsample": uniform(0.6, 0.4),               # 0.6 - 1.0 of rows per tree
    "model__colsample_bytree": uniform(0.5, 0.5),        # 0.5 - 1.0 of features per tree
    "model__reg_lambda": loguniform(1e-2, 1e2),          # regularisation
    "model__scale_pos_weight": [1.0, pos_weight],        # imbalance option A
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option B
}
xgb_res, xgb_pred = run_nested_cv(MODEL_NAME, xgb_pipe, xgb_params, X_train, y_train)
xgb_res.to_csv(OUT_DIR / "step05_XGB_folds.csv", index=False)
xgb_pred.to_csv(OUT_DIR / "step05_XGB_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.7 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(xgb_res, MODEL_NAME)
bp = xgb_res["best_params"]
n_spw = sum("'model__scale_pos_weight': 1.0" not in s for s in bp)
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, scale_pos_weight (balanced) {n_spw}")
print("Saved: out/step05_XGB_folds.csv")
print("=" * 66)

xgboost 3.1.3
WORKING... XGBoost is running. Please wait (about 40-90 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.239  ROC-AUC 0.869  recall 0.90  accuracy 0.69  (27s)
  fold  2/15  PR-AUC 0.171  ROC-AUC 0.837  recall 0.76  accuracy 0.75  (67s)
  fold  3/15  PR-AUC 0.155  ROC-AUC 0.830  recall 0.85  accuracy 0.69  (104s)
  fold  4/15  PR-AUC 0.168  ROC-AUC 0.838  recall 0.85  accuracy 0.69  (139s)
  fold  5/15  PR-AUC 0.187  ROC-AUC 0.831  recall 0.78  accuracy 0.72  (184s)
  fold  6/15  PR-AUC 0.218  ROC-AUC 0.853  recall 0.85  accuracy 0.71  (254s)
  fold  7/15  PR-AUC 0.188  ROC-AUC 0.839  recall 0.84  accuracy 0.69  (308s)
  fold  8/15  PR-AUC 0.176  ROC-AUC 0.858  recall 0.87  accuracy 0.70  (358s)
  fold  9/15  PR-AUC 0.175  ROC-AUC 0.841  recall 0.86  accuracy 0.69  (401s)
  fold 10/15  PR-AUC 0.179  ROC-AUC 0.837  recall 0.82  accuracy 0.70  (464s)
  fold 11/15  PR-AUC 0.173  ROC-AUC 0.829  recall 0.77

In [5]:
# ============================================================
# STEP 5.8  LIGHTGBM - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# Another boosting model (Microsoft); grows trees leaf-by-leaf, usually very fast.
# Needs: Cells 3.1, 4.0, 4.1 run in this session. Needs the 'lightgbm' package.
# ============================================================
from scipy.stats import randint, uniform
import lightgbm
from lightgbm import LGBMClassifier

MODEL_NAME = "LightGBM"
print(f"lightgbm {lightgbm.__version__}")
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 15-60 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

lgb_pipe = build_model_pipeline(
    LGBMClassifier(random_state=SEED, n_jobs=1, verbose=-1, subsample_freq=1),
    NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
lgb_params = {
    "model__n_estimators": randint(100, 601),
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__num_leaves": randint(8, 65),                 # complexity of each tree
    "model__max_depth": [-1, 3, 4, 6, 8, 10],            # -1 = no limit
    "model__min_child_samples": randint(10, 101),
    "model__subsample": uniform(0.6, 0.4),               # 0.6 - 1.0 of rows per tree
    "model__colsample_bytree": uniform(0.5, 0.5),        # 0.5 - 1.0 of features per tree
    "model__reg_lambda": loguniform(1e-2, 1e2),
    "model__class_weight": [None, "balanced"],           # imbalance option A
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option B
}
lgb_res, lgb_pred = run_nested_cv(MODEL_NAME, lgb_pipe, lgb_params, X_train, y_train)
lgb_res.to_csv(OUT_DIR / "step05_LGBM_folds.csv", index=False)
lgb_pred.to_csv(OUT_DIR / "step05_LGBM_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.8 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(lgb_res, MODEL_NAME)
bp = lgb_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced {bp.str.contains(chr(39) + 'balanced' + chr(39)).sum()}")
print("Saved: out/step05_LGBM_folds.csv")
print("=" * 66)

lightgbm 4.6.0
WORKING... LightGBM is running. Please wait (about 15-60 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.234  ROC-AUC 0.867  recall 0.84  accuracy 0.75  (31s)
  fold  2/15  PR-AUC 0.168  ROC-AUC 0.841  recall 0.87  accuracy 0.68  (86s)
  fold  3/15  PR-AUC 0.168  ROC-AUC 0.832  recall 0.81  accuracy 0.72  (160s)
  fold  4/15  PR-AUC 0.164  ROC-AUC 0.836  recall 0.84  accuracy 0.70  (223s)
  fold  5/15  PR-AUC 0.185  ROC-AUC 0.828  recall 0.85  accuracy 0.67  (282s)
  fold  6/15  PR-AUC 0.208  ROC-AUC 0.843  recall 0.88  accuracy 0.64  (343s)
  fold  7/15  PR-AUC 0.198  ROC-AUC 0.839  recall 0.81  accuracy 0.72  (404s)
  fold  8/15  PR-AUC 0.176  ROC-AUC 0.854  recall 0.86  accuracy 0.71  (473s)
  fold  9/15  PR-AUC 0.181  ROC-AUC 0.844  recall 0.83  accuracy 0.71  (549s)
  fold 10/15  PR-AUC 0.178  ROC-AUC 0.837  recall 0.83  accuracy 0.70  (635s)
  fold 11/15  PR-AUC 0.181  ROC-AUC 0.833  recall 0.

In [7]:
# ============================================================
# STEP 5.9  CATBOOST - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
# A third boosting model (Yandex); builds balanced ("symmetric") trees, robust defaults.
# Needs: Cells 3.1, 4.0, 4.1 run in this session. Needs the 'catboost' package.
# (v2: fixed - CatBoost needs the text "None", not Python None, for "no class weights")
# ============================================================
from scipy.stats import randint
import catboost
from catboost import CatBoostClassifier

MODEL_NAME = "CatBoost"
print(f"catboost {catboost.__version__}")
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 30-90 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

cb_pipe = build_model_pipeline(
    CatBoostClassifier(random_seed=SEED, thread_count=1, verbose=0, allow_writing_files=False),
    NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=False)
cb_params = {
    "model__iterations": randint(200, 801),              # number of boosting rounds
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__depth": randint(3, 9),                       # depth of each tree
    "model__l2_leaf_reg": loguniform(1, 30),             # regularisation
    "model__auto_class_weights": ["None", "Balanced"],   # imbalance option A ("None" as text: CatBoost rule)
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option B
}
cb_res, cb_pred = run_nested_cv(MODEL_NAME, cb_pipe, cb_params, X_train, y_train)
cb_res.to_csv(OUT_DIR / "step05_CatBoost_folds.csv", index=False)
cb_pred.to_csv(OUT_DIR / "step05_CatBoost_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.9 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(cb_res, MODEL_NAME)
bp = cb_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"Balanced {bp.str.contains(chr(39) + 'Balanced' + chr(39)).sum()}")
print("Saved: out/step05_CatBoost_folds.csv")
print("=" * 66)

catboost 1.2.8
WORKING... CatBoost is running. Please wait (about 30-90 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.227  ROC-AUC 0.869  recall 0.92  accuracy 0.66  (91s)
  fold  2/15  PR-AUC 0.174  ROC-AUC 0.845  recall 0.89  accuracy 0.68  (355s)
  fold  3/15  PR-AUC 0.170  ROC-AUC 0.836  recall 0.88  accuracy 0.68  (566s)
  fold  4/15  PR-AUC 0.166  ROC-AUC 0.835  recall 0.84  accuracy 0.69  (883s)
  fold  5/15  PR-AUC 0.187  ROC-AUC 0.833  recall 0.85  accuracy 0.69  (1100s)
  fold  6/15  PR-AUC 0.200  ROC-AUC 0.852  recall 0.82  accuracy 0.73  (1296s)
  fold  7/15  PR-AUC 0.182  ROC-AUC 0.840  recall 0.83  accuracy 0.71  (1486s)
  fold  8/15  PR-AUC 0.174  ROC-AUC 0.855  recall 0.86  accuracy 0.71  (1651s)
  fold  9/15  PR-AUC 0.178  ROC-AUC 0.842  recall 0.83  accuracy 0.73  (1863s)
  fold 10/15  PR-AUC 0.177  ROC-AUC 0.838  recall 0.79  accuracy 0.74  (2070s)
  fold 11/15  PR-AUC 0.171  ROC-AUC 0.833  re

In [8]:
# ============================================================
# STEP 5.10  SVM (RBF kernel, Nystroem approximation) - nested CV
# Same engine, same 15 folds, same tuning budget and imbalance options.
#
# Why an approximation: an exact RBF-kernel SVM grows with the SQUARE of the number of
# people (~17,600 per training fold) and would need days for 1,400 fits on a laptop.
# The Nystroem method (Williams & Seeger, 2001) builds an approximate RBF feature map and
# then trains a fast linear SVM on it - a standard, citable approach for large datasets.
# SVMs give scores, not probabilities -> a sigmoid (Platt) layer inside the model converts
# them to probabilities, fitted on the training fold only.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

MODEL_NAME = "SVM (RBF, Nystroem)"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 30-90 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

svm_model = Pipeline([
    ("rbf", Nystroem(kernel="rbf", random_state=SEED)),
    ("svm", CalibratedClassifierCV(LinearSVC(dual="auto", max_iter=5000, random_state=SEED),
                                   method="sigmoid", cv=3)),
])
svm_pipe = build_model_pipeline(svm_model, NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
svm_params = {
    "model__rbf__gamma": loguniform(1e-3, 1e0),                       # width of the RBF kernel
    "model__rbf__n_components": [300, 500, 800],                      # quality of the approximation
    "model__svm__estimator__C": loguniform(1e-3, 1e1),                # margin softness
    "model__svm__estimator__class_weight": [None, "balanced"],        # imbalance option A
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],     # imbalance option B
}
svm_res, svm_pred = run_nested_cv(MODEL_NAME, svm_pipe, svm_params, X_train, y_train)
svm_res.to_csv(OUT_DIR / "step05_SVM_folds.csv", index=False)
svm_pred.to_csv(OUT_DIR / "step05_SVM_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.10 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(svm_res, MODEL_NAME)
bp = svm_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced {bp.str.contains(chr(39) + 'balanced' + chr(39)).sum()}")
print("Saved: out/step05_SVM_folds.csv")
print("=" * 66)

WORKING... SVM (RBF, Nystroem) is running. Please wait (about 30-90 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.229  ROC-AUC 0.863  recall 0.84  accuracy 0.76  (75s)
  fold  2/15  PR-AUC 0.167  ROC-AUC 0.823  recall 0.71  accuracy 0.77  (286s)
  fold  3/15  PR-AUC 0.176  ROC-AUC 0.841  recall 0.82  accuracy 0.72  (458s)
  fold  4/15  PR-AUC 0.175  ROC-AUC 0.839  recall 0.81  accuracy 0.72  (552s)
  fold  5/15  PR-AUC 0.198  ROC-AUC 0.837  recall 0.78  accuracy 0.75  (678s)
  fold  6/15  PR-AUC 0.211  ROC-AUC 0.853  recall 0.81  accuracy 0.74  (828s)
  fold  7/15  PR-AUC 0.192  ROC-AUC 0.837  recall 0.80  accuracy 0.74  (962s)
  fold  8/15  PR-AUC 0.173  ROC-AUC 0.856  recall 0.85  accuracy 0.73  (1183s)
  fold  9/15  PR-AUC 0.183  ROC-AUC 0.840  recall 0.76  accuracy 0.75  (1325s)
  fold 10/15  PR-AUC 0.174  ROC-AUC 0.835  recall 0.82  accuracy 0.73  (1491s)
  fold 11/15  PR-AUC 0.162  ROC-AUC 0.830  recall 0.

In [9]:
# ============================================================
# STEP 5.11  K-NEAREST NEIGHBOURS (KNN) - nested CV
# Same engine, same 15 folds, same tuning budget.
# A person's risk = share of strokes among the K most similar people in the training fold.
# KNN has no class_weight option -> the only imbalance option is SMOTE (or none).
# Large K values are allowed because strokes are rare (4%).
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from scipy.stats import randint
from sklearn.neighbors import KNeighborsClassifier

MODEL_NAME = "KNN"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 20-60 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

knn_pipe = build_model_pipeline(KNeighborsClassifier(n_jobs=1),
                                NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
knn_params = {
    "model__n_neighbors": randint(5, 301),               # how many similar people to look at
    "model__weights": ["uniform", "distance"],           # closer people count more?
    "model__p": [1, 2],                                  # distance type: Manhattan or Euclidean
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option
}
knn_res, knn_pred = run_nested_cv(MODEL_NAME, knn_pipe, knn_params, X_train, y_train)
knn_res.to_csv(OUT_DIR / "step05_KNN_folds.csv", index=False)
knn_pred.to_csv(OUT_DIR / "step05_KNN_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.11 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(knn_res, MODEL_NAME)
bp = knn_res["best_params"]
print(f"\nSettings chosen (out of 15 folds): SMOTE {bp.str.contains('FunctionSampler').sum()}")
print("Saved: out/step05_KNN_folds.csv")
print("=" * 66)

WORKING... KNN is running. Please wait (about 20-60 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.201  ROC-AUC 0.855  recall 0.86  accuracy 0.70  (58s)
  fold  2/15  PR-AUC 0.152  ROC-AUC 0.817  recall 0.78  accuracy 0.72  (123s)
  fold  3/15  PR-AUC 0.144  ROC-AUC 0.814  recall 0.80  accuracy 0.72  (246s)
  fold  4/15  PR-AUC 0.159  ROC-AUC 0.832  recall 0.76  accuracy 0.75  (338s)
  fold  5/15  PR-AUC 0.182  ROC-AUC 0.816  recall 0.83  accuracy 0.71  (441s)
  fold  6/15  PR-AUC 0.186  ROC-AUC 0.830  recall 0.83  accuracy 0.68  (537s)
  fold  7/15  PR-AUC 0.170  ROC-AUC 0.820  recall 0.75  accuracy 0.74  (637s)
  fold  8/15  PR-AUC 0.163  ROC-AUC 0.838  recall 0.80  accuracy 0.72  (716s)
  fold  9/15  PR-AUC 0.171  ROC-AUC 0.825  recall 0.87  accuracy 0.68  (812s)
  fold 10/15  PR-AUC 0.137  ROC-AUC 0.810  recall 0.80  accuracy 0.70  (887s)
  fold 11/15  PR-AUC 0.165  ROC-AUC 0.820  recall 0.83  accuracy 0.67  

In [10]:
# ============================================================
# STEP 5.12  NAIVE BAYES (Gaussian) - nested CV
# Same engine, same 15 folds, same tuning budget.
# A simple probability model that assumes the features are independent of each other.
# Imbalance option A for Naive Bayes = equal class priors (its version of "balanced").
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from sklearn.naive_bayes import GaussianNB

MODEL_NAME = "Naive Bayes"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 5-20 min). "
      "A line appears after each of the 15 folds.", flush=True)

nb_pipe = build_model_pipeline(GaussianNB(), NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
nb_params = {
    "model__var_smoothing": loguniform(1e-12, 1e-1),     # stabilises the variance estimates
    "model__priors": [None, [0.5, 0.5]],                 # imbalance option A (equal priors)
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option B
}
nb_res, nb_pred = run_nested_cv(MODEL_NAME, nb_pipe, nb_params, X_train, y_train)
nb_res.to_csv(OUT_DIR / "step05_NB_folds.csv", index=False)
nb_pred.to_csv(OUT_DIR / "step05_NB_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.12 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(nb_res, MODEL_NAME)
bp = nb_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"equal priors {bp.str.contains('[0.5, 0.5]', regex=False).sum()}")
print("Saved: out/step05_NB_folds.csv")
print("=" * 66)

WORKING... Naive Bayes is running. Please wait (about 5-20 min). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.154  ROC-AUC 0.840  recall 0.87  accuracy 0.68  (16s)
  fold  2/15  PR-AUC 0.118  ROC-AUC 0.794  recall 0.80  accuracy 0.68  (31s)
  fold  3/15  PR-AUC 0.153  ROC-AUC 0.817  recall 0.84  accuracy 0.66  (50s)
  fold  4/15  PR-AUC 0.125  ROC-AUC 0.820  recall 0.86  accuracy 0.67  (65s)
  fold  5/15  PR-AUC 0.144  ROC-AUC 0.805  recall 0.83  accuracy 0.65  (84s)
  fold  6/15  PR-AUC 0.157  ROC-AUC 0.829  recall 0.87  accuracy 0.66  (99s)
  fold  7/15  PR-AUC 0.133  ROC-AUC 0.809  recall 0.86  accuracy 0.65  (118s)
  fold  8/15  PR-AUC 0.137  ROC-AUC 0.833  recall 0.85  accuracy 0.69  (139s)
  fold  9/15  PR-AUC 0.154  ROC-AUC 0.831  recall 0.83  accuracy 0.69  (157s)
  fold 10/15  PR-AUC 0.144  ROC-AUC 0.813  recall 0.78  accuracy 0.72  (184s)
  fold 11/15  PR-AUC 0.132  ROC-AUC 0.806  recall 0.79  accuracy 0.68  (215s)
  fold 12/15  PR-AUC 0.152  ROC-AUC 0.82

In [11]:
# ============================================================
# STEP 5.13  MLP (small neural network) - nested CV
# Same engine, same 15 folds, same tuning budget.
# A feed-forward neural network with 1-2 hidden layers.
# MLP has no class_weight option -> the only imbalance option is SMOTE (or none).
# early_stopping holds out 10% OF THE TRAINING FOLD to decide when to stop -> no leakage.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from sklearn.neural_network import MLPClassifier

MODEL_NAME = "MLP (neural network)"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 30-90 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

mlp_pipe = build_model_pipeline(
    MLPClassifier(max_iter=300, early_stopping=True, validation_fraction=0.1,
                  n_iter_no_change=10, random_state=SEED),
    NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
mlp_params = {
    "model__hidden_layer_sizes": [(32,), (64,), (128,), (64, 32), (128, 64)],   # network size
    "model__alpha": loguniform(1e-5, 1e-1),              # regularisation (prevents overfitting)
    "model__learning_rate_init": loguniform(1e-4, 1e-2), # learning speed
    "model__batch_size": [64, 128, 256],
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option
}
mlp_res, mlp_pred = run_nested_cv(MODEL_NAME, mlp_pipe, mlp_params, X_train, y_train)
mlp_res.to_csv(OUT_DIR / "step05_MLP_folds.csv", index=False)
mlp_pred.to_csv(OUT_DIR / "step05_MLP_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.13 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(mlp_res, MODEL_NAME)
bp = mlp_res["best_params"]
print(f"\nSettings chosen (out of 15 folds): SMOTE {bp.str.contains('FunctionSampler').sum()}")
print("Saved: out/step05_MLP_folds.csv")
print("=" * 66)

WORKING... MLP (neural network) is running. Please wait (about 30-90 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.223  ROC-AUC 0.862  recall 0.91  accuracy 0.66  (94s)
  fold  2/15  PR-AUC 0.163  ROC-AUC 0.824  recall 0.62  accuracy 0.81  (217s)
  fold  3/15  PR-AUC 0.173  ROC-AUC 0.830  recall 0.82  accuracy 0.71  (323s)
  fold  4/15  PR-AUC 0.147  ROC-AUC 0.825  recall 0.88  accuracy 0.63  (470s)
  fold  5/15  PR-AUC 0.192  ROC-AUC 0.834  recall 0.79  accuracy 0.73  (603s)
  fold  6/15  PR-AUC 0.199  ROC-AUC 0.848  recall 0.83  accuracy 0.72  (752s)
  fold  7/15  PR-AUC 0.161  ROC-AUC 0.838  recall 0.79  accuracy 0.75  (870s)
  fold  8/15  PR-AUC 0.177  ROC-AUC 0.850  recall 0.91  accuracy 0.67  (1018s)
  fold  9/15  PR-AUC 0.143  ROC-AUC 0.810  recall 0.83  accuracy 0.68  (1129s)
  fold 10/15  PR-AUC 0.169  ROC-AUC 0.830  recall 0.77  accuracy 0.76  (1295s)
  fold 11/15  PR-AUC 0.154  ROC-AUC 0.832  recall 0

In [12]:
# ============================================================
# STEP 5.14  SOFT VOTING ENSEMBLE - nested CV
# Same engine, same 15 folds, same tuning budget (30 candidates) and imbalance options.
#
# Members = three DIFFERENT kinds of model from the top group, so their errors differ:
#   - Logistic Regression     (linear)
#   - XGBoost                 (boosted trees)
#   - SVM (RBF, Nystroem)     (kernel method)
# Soft voting = average of the members' predicted probabilities (optionally weighted).
# All members are re-tuned INSIDE each outer training fold (joint random search) -> no leakage,
# and no settings are copied from earlier results.
# Needs: Cells 3.1, 4.0, 4.1 run in this session. Needs 'xgboost'.
# ============================================================
from scipy.stats import randint
from sklearn.ensemble import VotingClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

MODEL_NAME = "Soft Voting (LR + XGB + SVM)"
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 60-120 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

vote_model = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(solver="liblinear", max_iter=5000)),
        ("xgb", XGBClassifier(tree_method="hist", eval_metric="logloss", random_state=SEED,
                              n_jobs=1, verbosity=0)),
        ("svm", Pipeline([
            ("rbf", Nystroem(kernel="rbf", n_components=500, random_state=SEED)),
            ("svm", CalibratedClassifierCV(LinearSVC(dual="auto", max_iter=5000, random_state=SEED),
                                           method="sigmoid", cv=3)),
        ])),
    ],
    voting="soft", n_jobs=1)
vote_pipe = build_model_pipeline(vote_model, NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
vote_params = {
    # Logistic Regression member
    "model__lr__C": loguniform(1e-3, 1e2),
    "model__lr__penalty": ["l1", "l2"],
    "model__lr__class_weight": [None, "balanced"],
    # XGBoost member
    "model__xgb__n_estimators": randint(100, 601),
    "model__xgb__learning_rate": loguniform(0.01, 0.3),
    "model__xgb__max_depth": randint(2, 9),
    "model__xgb__scale_pos_weight": [1.0, pos_weight],
    # SVM member
    "model__svm__rbf__gamma": loguniform(1e-3, 1e0),
    "model__svm__svm__estimator__C": loguniform(1e-3, 1e1),
    # how much each member counts
    "model__weights": [[1, 1, 1], [2, 1, 1], [1, 2, 1], [1, 1, 2]],
    # shared imbalance option
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
vote_res, vote_pred = run_nested_cv(MODEL_NAME, vote_pipe, vote_params, X_train, y_train)
vote_res.to_csv(OUT_DIR / "step05_Voting_folds.csv", index=False)
vote_pred.to_csv(OUT_DIR / "step05_Voting_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.14 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(vote_res, MODEL_NAME)
bp = vote_res["best_params"]
print(f"\nSettings chosen (out of 15 folds): SMOTE {bp.str.contains('FunctionSampler').sum()}")
for w in ["[1, 1, 1]", "[2, 1, 1]", "[1, 2, 1]", "[1, 1, 2]"]:
    print(f"  weights {w} (LR, XGB, SVM): {bp.str.contains(w, regex=False).sum()} folds")
print("Saved: out/step05_Voting_folds.csv")
print("=" * 66)

WORKING... Soft Voting (LR + XGB + SVM) is running. Please wait (about 60-120 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.241  ROC-AUC 0.868  recall 0.87  accuracy 0.74  (112s)
  fold  2/15  PR-AUC 0.174  ROC-AUC 0.842  recall 0.80  accuracy 0.74  (370s)
  fold  3/15  PR-AUC 0.189  ROC-AUC 0.841  recall 0.85  accuracy 0.70  (618s)
  fold  4/15  PR-AUC 0.173  ROC-AUC 0.835  recall 0.82  accuracy 0.70  (834s)
  fold  5/15  PR-AUC 0.193  ROC-AUC 0.835  recall 0.78  accuracy 0.75  (1049s)
  fold  6/15  PR-AUC 0.224  ROC-AUC 0.855  recall 0.84  accuracy 0.72  (1223s)
  fold  7/15  PR-AUC 0.183  ROC-AUC 0.844  recall 0.85  accuracy 0.69  (1433s)
  fold  8/15  PR-AUC 0.178  ROC-AUC 0.857  recall 0.90  accuracy 0.69  (1653s)
  fold  9/15  PR-AUC 0.175  ROC-AUC 0.847  recall 0.78  accuracy 0.76  (1911s)
  fold 10/15  PR-AUC 0.173  ROC-AUC 0.837  recall 0.85  accuracy 0.70  (2137s)
  fold 11/15  PR-AUC 0.187  ROC-AUC 0.

In [13]:
# ============================================================
# STEP 5.15  STACKING ENSEMBLE - nested CV
# Same engine, same 15 folds, same tuning budget (30 candidates) and imbalance options.
#
# Same three members as soft voting (LR + XGBoost + SVM), but instead of averaging,
# a small Logistic Regression ("meta-model") LEARNS how to combine their predictions.
# The meta-model is trained on out-of-fold member predictions made INSIDE the training
# fold (StackingClassifier cv=3) -> it never sees its own training rows twice. No leakage.
# This is the slowest model (members are trained 4x per candidate). Best run overnight.
# Needs: Cells 3.1, 4.0, 4.1 run in this session. Needs 'xgboost'.
# ============================================================
from scipy.stats import randint
from sklearn.ensemble import StackingClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

MODEL_NAME = "Stacking (LR + XGB + SVM -> LR)"
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 3-5 HOURS - best overnight; "
      "laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.",
      flush=True)

stack_model = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(solver="liblinear", max_iter=5000)),
        ("xgb", XGBClassifier(tree_method="hist", eval_metric="logloss", random_state=SEED,
                              n_jobs=1, verbosity=0)),
        ("svm", Pipeline([
            ("rbf", Nystroem(kernel="rbf", n_components=500, random_state=SEED)),
            ("svm", CalibratedClassifierCV(LinearSVC(dual="auto", max_iter=5000, random_state=SEED),
                                           method="sigmoid", cv=3)),
        ])),
    ],
    final_estimator=LogisticRegression(max_iter=5000),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    stack_method="predict_proba", n_jobs=1)
stack_pipe = build_model_pipeline(stack_model, NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
stack_params = {
    # Logistic Regression member
    "model__lr__C": loguniform(1e-3, 1e2),
    "model__lr__penalty": ["l1", "l2"],
    "model__lr__class_weight": [None, "balanced"],
    # XGBoost member
    "model__xgb__n_estimators": randint(100, 601),
    "model__xgb__learning_rate": loguniform(0.01, 0.3),
    "model__xgb__max_depth": randint(2, 9),
    "model__xgb__scale_pos_weight": [1.0, pos_weight],
    # SVM member
    "model__svm__rbf__gamma": loguniform(1e-3, 1e0),
    "model__svm__svm__estimator__C": loguniform(1e-3, 1e1),
    # meta-model (combiner)
    "model__final_estimator__C": loguniform(1e-2, 1e2),
    # shared imbalance option
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],
}
stack_res, stack_pred = run_nested_cv(MODEL_NAME, stack_pipe, stack_params, X_train, y_train)
stack_res.to_csv(OUT_DIR / "step05_Stacking_folds.csv", index=False)
stack_pred.to_csv(OUT_DIR / "step05_Stacking_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.15 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(stack_res, MODEL_NAME)
bp = stack_res["best_params"]
print(f"\nSettings chosen (out of 15 folds): SMOTE {bp.str.contains('FunctionSampler').sum()}")
print("Saved: out/step05_Stacking_folds.csv")
print("=" * 66)

WORKING... Stacking (LR + XGB + SVM -> LR) is running. Please wait (about 3-5 HOURS - best overnight; laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.223  ROC-AUC 0.865  recall 0.88  accuracy 0.71  (492s)
  fold  2/15  PR-AUC 0.164  ROC-AUC 0.840  recall 0.82  accuracy 0.73  (1101s)
  fold  3/15  PR-AUC 0.172  ROC-AUC 0.840  recall 0.81  accuracy 0.72  (1619s)
  fold  4/15  PR-AUC 0.175  ROC-AUC 0.837  recall 0.83  accuracy 0.70  (2086s)
  fold  5/15  PR-AUC 0.194  ROC-AUC 0.834  recall 0.75  accuracy 0.76  (2672s)
  fold  6/15  PR-AUC 0.220  ROC-AUC 0.850  recall 0.81  accuracy 0.75  (3307s)
  fold  7/15  PR-AUC 0.200  ROC-AUC 0.842  recall 0.82  accuracy 0.73  (3829s)
  fold  8/15  PR-AUC 0.173  ROC-AUC 0.856  recall 0.83  accuracy 0.73  (4289s)
  fold  9/15  PR-AUC 0.178  ROC-AUC 0.848  recall 0.89  accuracy 0.67  (4763s)
  fold 10/15  PR-AUC 0.170  ROC-AUC 0.834  recall 0.78  accuracy 0.74  (5231s)
  fold 11/15  PR-

In [14]:
# ============================================================
# STEP 5.16  OUR OWN MODEL: Stability-Selected Parsimonious Logistic Regression (SSP-LR)
# Same engine, same 15 folds, same tuning budget (30 candidates) and imbalance options.
#
# WHY THIS MODEL (rationale for the thesis):
#   Steps 4-5 showed that complex models do NOT beat Logistic Regression on NHANES.
#   So the useful contribution is not "more complex", but "simpler AND more trustworthy":
#   a short list of risk factors that is STABLE (re-appears when the data change) - this
#   directly serves our gap G3 (explanations that are validated, not taken on trust).
#
# HOW IT WORKS (inside every training fold only -> no leakage):
#   1. Draw 30 random half-samples of the training fold.
#   2. On each half-sample fit a LASSO (L1) logistic regression -> it keeps only some features.
#   3. Count how often each feature is kept ("selection frequency", 0-100%).
#   4. Keep only features kept in >= threshold of the half-samples (e.g. 60%) = STABLE features.
#   5. Fit an ordinary (L2) logistic regression on the stable features only.
# Based on "stability selection" (Meinshausen & Buhlmann, 2010, J R Stat Soc B).
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted


class StabilityLR(ClassifierMixin, BaseEstimator):
    """Stability-selected parsimonious logistic regression (our model)."""

    def __init__(self, n_subsamples=30, sample_fraction=0.5, threshold=0.6,
                 C_select=0.05, C_final=1.0, class_weight=None, random_state=42):
        self.n_subsamples = n_subsamples
        self.sample_fraction = sample_fraction
        self.threshold = threshold
        self.C_select = C_select
        self.C_final = C_final
        self.class_weight = class_weight
        self.random_state = random_state

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        rng = np.random.RandomState(self.random_state)
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        counts = np.zeros(X.shape[1])
        for _ in range(self.n_subsamples):
            # stratified half-sample: keeps the 4% stroke rate in every subsample
            idx = np.concatenate([
                rng.choice(pos, int(len(pos) * self.sample_fraction), replace=False),
                rng.choice(neg, int(len(neg) * self.sample_fraction), replace=False)])
            lasso = LogisticRegression(penalty="l1", solver="liblinear", C=self.C_select,
                                       class_weight=self.class_weight, max_iter=5000)
            lasso.fit(X[idx], y[idx])
            counts += (np.abs(lasso.coef_[0]) > 1e-8)
        self.selection_freq_ = counts / self.n_subsamples
        self.selected_ = self.selection_freq_ >= self.threshold
        if not self.selected_.any():                       # safety: keep the most stable feature
            self.selected_[np.argmax(self.selection_freq_)] = True
        self.final_ = LogisticRegression(C=self.C_final, class_weight=self.class_weight, max_iter=5000)
        self.final_.fit(X[:, self.selected_], y)
        self.n_features_in_ = X.shape[1]
        return self

    def predict_proba(self, X):
        check_is_fitted(self, "final_")
        X = check_array(X)
        return self.final_.predict_proba(X[:, self.selected_])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


MODEL_NAME = "OWN: Stability-selected LR (SSP-LR)"
print(f"WORKING... {MODEL_NAME} is running. Please wait (about 40-100 min - laptop plugged in, "
      "'caffeinate -i' in Terminal). A line appears after each of the 15 folds.", flush=True)

own_pipe = build_model_pipeline(StabilityLR(random_state=SEED),
                                NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
own_params = {
    "model__threshold": [0.5, 0.6, 0.7, 0.8],            # how stable a feature must be to stay
    "model__C_select": loguniform(5e-3, 5e-1),           # how strict the LASSO step is
    "model__C_final": loguniform(1e-2, 1e2),             # regularisation of the final model
    "model__class_weight": [None, "balanced"],           # imbalance option A
    "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))],   # imbalance option B
}
own_res, own_pred = run_nested_cv(MODEL_NAME, own_pipe, own_params, X_train, y_train)
own_res.to_csv(OUT_DIR / "step05_OWN_SSPLR_folds.csv", index=False)
own_pred.to_csv(OUT_DIR / "step05_OWN_SSPLR_outer_predictions.csv", index=False)

print("\nDONE - all 15 folds finished.\n")
print("=" * 66)
print(f"STEP 5.16 SUMMARY - {MODEL_NAME}")
print("=" * 66)
summarise(own_res, MODEL_NAME)
bp = own_res["best_params"]
print("\nSettings chosen (out of 15 folds): "
      f"SMOTE {bp.str.contains('FunctionSampler').sum()}, "
      f"balanced {bp.str.contains(chr(39) + 'balanced' + chr(39)).sum()}")
for t in [0.5, 0.6, 0.7, 0.8]:
    print(f"  stability threshold {t}: {bp.str.contains(f'threshold{chr(39)}: {t}', regex=False).sum()} folds")
print("Saved: out/step05_OWN_SSPLR_folds.csv")
print("=" * 66)

WORKING... OWN: Stability-selected LR (SSP-LR) is running. Please wait (about 40-100 min - laptop plugged in, 'caffeinate -i' in Terminal). A line appears after each of the 15 folds.
  fold  1/15  PR-AUC 0.215  ROC-AUC 0.867  recall 0.87  accuracy 0.75  (156s)
  fold  2/15  PR-AUC 0.179  ROC-AUC 0.843  recall 0.82  accuracy 0.73  (404s)
  fold  3/15  PR-AUC 0.176  ROC-AUC 0.844  recall 0.82  accuracy 0.73  (626s)
  fold  4/15  PR-AUC 0.172  ROC-AUC 0.835  recall 0.81  accuracy 0.71  (977s)
  fold  5/15  PR-AUC 0.187  ROC-AUC 0.835  recall 0.79  accuracy 0.74  (1190s)
  fold  6/15  PR-AUC 0.208  ROC-AUC 0.852  recall 0.85  accuracy 0.70  (1452s)
  fold  7/15  PR-AUC 0.207  ROC-AUC 0.844  recall 0.82  accuracy 0.70  (1736s)
  fold  8/15  PR-AUC 0.168  ROC-AUC 0.851  recall 0.83  accuracy 0.72  (2055s)
  fold  9/15  PR-AUC 0.183  ROC-AUC 0.849  recall 0.79  accuracy 0.75  (2274s)
  fold 10/15  PR-AUC 0.170  ROC-AUC 0.833  recall 0.80  accuracy 0.73  (2629s)
  fold 11/15  PR-AUC 0.173  ROC

In [15]:
# ============================================================
# STEP 6.1  STATISTICAL COMPARISON OF ALL MODELS  (runs in seconds)
# Reads the saved fold results of Steps 4-5 (TRAIN only, same 15 paired folds).
#
#  a) Friedman test: do the models differ at all? (non-parametric, across the 15 folds)
#  b) Each model vs the best one: corrected resampled t-test (Nadeau & Bengio, 2003).
#     The correction is needed because CV folds share training data, so a normal t-test
#     would be over-confident. p-values are adjusted for 15 comparisons with Holm's method.
#  c) Selection rule (fixed in the analysis plan): choose the SIMPLEST model that is NOT
#     significantly worse than the best model on the primary metric (PR-AUC).
#     Complexity order is set below, before looking at the test results.
# Needs: Cell 3.1 run in this session (for OUT_DIR). Does not use the TEST set.
# ============================================================
import numpy as np
import pandas as pd
from scipy import stats

FILES = {   # file -> (short name, complexity rank: 1 = simplest)
    "step05_OWN_SSPLR_folds.csv":    ("SSP-LR (own model)", 1),   # LR on a reduced, stable feature set
    "step04_LR_folds.csv":           ("Logistic Regression", 2),
    "step05_LR_splines_folds.csv":   ("LR + splines", 3),
    "step05_NB_folds.csv":           ("Naive Bayes", 3),
    "step05_DT_folds.csv":           ("Decision Tree", 3),
    "step05_KNN_folds.csv":          ("KNN", 4),
    "step05_SVM_folds.csv":          ("SVM (Nystroem)", 5),
    "step05_RF_folds.csv":           ("Random Forest", 6),
    "step05_ET_folds.csv":           ("Extra Trees", 6),
    "step05_BRF_folds.csv":          ("Balanced RF", 6),
    "step05_XGB_folds.csv":          ("XGBoost", 7),
    "step05_LGBM_folds.csv":         ("LightGBM", 7),
    "step05_CatBoost_folds.csv":     ("CatBoost", 7),
    "step05_MLP_folds.csv":          ("MLP", 7),
    "step05_Voting_folds.csv":       ("Soft Voting", 8),
    "step05_Stacking_folds.csv":     ("Stacking", 9),
}
PRIMARY = "pr_auc"          # primary metric (also the tuning metric)
ALPHA = 0.05
N_FOLDS, K = 15, 5          # 15 outer folds from 5-fold CV repeated 3x
TEST_TRAIN_RATIO = (1 / K) / (1 - 1 / K)   # = 0.25 (validation size / training size)

# ---- load -----------------------------------------------------------------------------
data, missing = {}, []
for f, (name, rank) in FILES.items():
    p = OUT_DIR / f
    if not p.exists():
        missing.append(f); continue
    d = pd.read_csv(p).sort_values("fold")
    assert len(d) == N_FOLDS, f"{f} does not have 15 folds"
    data[name] = d
if missing:
    print("WARNING - missing result files (these models are left out):", missing)

names = list(data)
RANK = {name: rank for name, rank in FILES.values()}
P = pd.DataFrame({n: data[n][PRIMARY].values for n in names})          # 15 x models
R = pd.DataFrame({n: data[n]["roc_auc"].values for n in names})
REC = pd.DataFrame({n: data[n]["youden_recall"].values for n in names})
ACC = pd.DataFrame({n: data[n]["youden_accuracy"].values for n in names})

# ---- a) Friedman ----------------------------------------------------------------------
fr_stat, fr_p = stats.friedmanchisquare(*[P[n].values for n in names])
mean_rank = P.rank(axis=1, ascending=False).mean().sort_values()     # 1 = best per fold

# ---- b) corrected resampled t-test vs the best model ----------------------------------
best = P.mean().idxmax()
rows = []
for n in names:
    diff = P[best].values - P[n].values                 # > 0 means the best model is better
    if n == best:
        t, p = 0.0, 1.0
    else:
        var = diff.var(ddof=1)
        se = np.sqrt((1 / N_FOLDS + TEST_TRAIN_RATIO) * var) if var > 0 else np.nan
        t = diff.mean() / se if se and se > 0 else 0.0
        p = 2 * stats.t.sf(abs(t), df=N_FOLDS - 1)
    rows.append(dict(model=n, complexity=RANK[n],
                     pr_auc=P[n].mean(), pr_auc_sd=P[n].std(), roc_auc=R[n].mean(),
                     recall=REC[n].mean(), accuracy=ACC[n].mean(), mean_rank=mean_rank[n],
                     diff_vs_best=diff.mean(), t=t, p_raw=p))
cmp = pd.DataFrame(rows)

# Holm adjustment over the comparisons with the best model
others = cmp[cmp.model != best].sort_values("p_raw")
m = len(others)
holm, running = {}, 0.0
for i, (_, r) in enumerate(others.iterrows()):
    running = max(running, min(1.0, (m - i) * r["p_raw"]))
    holm[r["model"]] = running
cmp["p_holm"] = cmp["model"].map(holm).fillna(1.0)
cmp["verdict"] = np.where(cmp.model == best, "BEST (reference)",
                          np.where(cmp.p_holm < ALPHA, "significantly worse", "equivalent to best"))
cmp = cmp.sort_values("pr_auc", ascending=False).reset_index(drop=True)
cmp.to_csv(OUT_DIR / "step06_model_comparison.csv", index=False)

# ---- c) selection rule ----------------------------------------------------------------
eligible = cmp[cmp.verdict != "significantly worse"].sort_values(["complexity", "pr_auc"],
                                                                  ascending=[True, False])
chosen = eligible.iloc[0]["model"]

# ---- print ----------------------------------------------------------------------------
print("=" * 70)
print("STEP 6.1 SUMMARY - STATISTICAL COMPARISON (15 paired folds, TRAIN only)")
print("=" * 70)
print(f"\n[A] FRIEDMAN TEST (do the {len(names)} models differ at all?)")
print(f"  chi-square = {fr_stat:.2f}, p = {fr_p:.2e} -> "
      + ("YES, at least one model differs" if fr_p < ALPHA else "NO clear difference"))

print(f"\n[B] EACH MODEL vs THE BEST ({best}), primary metric PR-AUC")
print("    (corrected resampled t-test, Holm-adjusted p)")
for _, r in cmp.iterrows():
    print(f"  {r['model']:<22} PR-AUC {r['pr_auc']:.3f}  ROC-AUC {r['roc_auc']:.3f}  "
          f"recall {100*r['recall']:4.1f}%  acc {100*r['accuracy']:4.1f}%  "
          f"p={r['p_holm']:.3f}  -> {r['verdict']}")

n_eq = (cmp.verdict == "equivalent to best").sum()
n_worse = (cmp.verdict == "significantly worse").sum()
print(f"\n  Equivalent to the best: {n_eq} models | significantly worse: {n_worse} models")

print("\n[C] SELECTION RULE: simplest model that is NOT significantly worse than the best")
print(f"  CHOSEN MODEL: {chosen}")
c = cmp[cmp.model == chosen].iloc[0]
print(f"  PR-AUC {c['pr_auc']:.3f} | ROC-AUC {c['roc_auc']:.3f} | recall {100*c['recall']:.1f}% | "
      f"accuracy {100*c['accuracy']:.1f}%")
print("\nSaved: out/step06_model_comparison.csv")
print("=" * 70)

STEP 6.1 SUMMARY - STATISTICAL COMPARISON (15 paired folds, TRAIN only)

[A] FRIEDMAN TEST (do the 16 models differ at all?)
  chi-square = 129.46, p = 2.71e-20 -> YES, at least one model differs

[B] EACH MODEL vs THE BEST (Soft Voting), primary metric PR-AUC
    (corrected resampled t-test, Holm-adjusted p)
  Soft Voting            PR-AUC 0.191  ROC-AUC 0.847  recall 83.5%  acc 71.7%  p=1.000  -> BEST (reference)
  Logistic Regression    PR-AUC 0.189  ROC-AUC 0.845  recall 81.9%  acc 72.5%  p=1.000  -> equivalent to best
  SVM (Nystroem)         PR-AUC 0.188  ROC-AUC 0.843  recall 80.6%  acc 73.9%  p=1.000  -> equivalent to best
  SSP-LR (own model)     PR-AUC 0.188  ROC-AUC 0.846  recall 82.1%  acc 72.2%  p=1.000  -> equivalent to best
  Stacking               PR-AUC 0.188  ROC-AUC 0.845  recall 82.7%  acc 71.9%  p=1.000  -> equivalent to best
  LR + splines           PR-AUC 0.188  ROC-AUC 0.845  recall 82.3%  acc 72.3%  p=1.000  -> equivalent to best
  XGBoost                PR-AUC

In [16]:
# ============================================================
# STEP 7.1  FIT THE FINAL MODEL ON ALL OF TRAIN  (TEST still locked)
#
# Final model  : SSP-LR (own model) - chosen in Step 6.1
# Comparator   : plain Logistic Regression (declared in the plan, as in the pilot)
#
# Exactly the same recipe as inside each outer fold, now on the whole TRAIN set:
#   1. tune with a 3-fold random search (30 candidates, PR-AUC)
#   2. out-of-fold predictions on TRAIN -> Platt calibration + thresholds (Youden, max-F2)
#   3. refit on all of TRAIN
# Then: show WHICH risk factors SSP-LR kept, how stable they are, and their odds ratios.
# Needs: Cells 3.1, 4.0, 4.1 run in this session.
# ============================================================
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted


# --- SSP-LR definition (same as Step 5.16; repeated so this cell works after a restart) ---
class StabilityLR(ClassifierMixin, BaseEstimator):
    """Stability-selected parsimonious logistic regression (our model)."""

    def __init__(self, n_subsamples=30, sample_fraction=0.5, threshold=0.6,
                 C_select=0.05, C_final=1.0, class_weight=None, random_state=42):
        self.n_subsamples = n_subsamples
        self.sample_fraction = sample_fraction
        self.threshold = threshold
        self.C_select = C_select
        self.C_final = C_final
        self.class_weight = class_weight
        self.random_state = random_state

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        rng = np.random.RandomState(self.random_state)
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        counts = np.zeros(X.shape[1])
        for _ in range(self.n_subsamples):
            idx = np.concatenate([
                rng.choice(pos, int(len(pos) * self.sample_fraction), replace=False),
                rng.choice(neg, int(len(neg) * self.sample_fraction), replace=False)])
            lasso = LogisticRegression(penalty="l1", solver="liblinear", C=self.C_select,
                                       class_weight=self.class_weight, max_iter=5000)
            lasso.fit(X[idx], y[idx])
            counts += (np.abs(lasso.coef_[0]) > 1e-8)
        self.selection_freq_ = counts / self.n_subsamples
        self.selected_ = self.selection_freq_ >= self.threshold
        if not self.selected_.any():
            self.selected_[np.argmax(self.selection_freq_)] = True
        self.final_ = LogisticRegression(C=self.C_final, class_weight=self.class_weight, max_iter=5000)
        self.final_.fit(X[:, self.selected_], y)
        self.n_features_in_ = X.shape[1]
        return self

    def predict_proba(self, X):
        check_is_fitted(self, "final_")
        X = check_array(X)
        return self.final_.predict_proba(X[:, self.selected_])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def encoded_feature_names(fitted_pipe, numeric, binary, categorical):
    """Readable names of the columns the model actually sees (after fill + one-hot)."""
    fill = fitted_pipe.named_steps["fill"]
    ord_cats = fill.named_transformers_["cat"].named_steps["ord"].categories_
    flag_idx = fill.named_transformers_["flags"].features_
    flag_names = [f"{(numeric + binary)[i]} (was missing)" for i in flag_idx]
    onehot = fitted_pipe.named_steps["encode"].named_transformers_["cat"]
    oh_names = []
    for j, cat in enumerate(categorical):
        for code in onehot.categories_[j]:
            label = ord_cats[j][int(code)] if 0 <= int(code) < len(ord_cats[j]) else "unknown"
            oh_names.append(f"{cat} = {label}")
    return list(numeric) + oh_names + list(binary) + flag_names


def fit_final(name, pipe, params):
    """Tune on all TRAIN, then calibration + thresholds from TRAIN out-of-fold predictions."""
    cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    search = RandomizedSearchCV(pipe, params, n_iter=N_ITER, scoring=TUNING_METRIC, cv=cv,
                                random_state=SEED, n_jobs=N_JOBS, refit=True)
    search.fit(X_train, y_train)
    oof = cross_val_predict(clone(search.best_estimator_), X_train, y_train, cv=cv,
                            method="predict_proba", n_jobs=N_JOBS)[:, 1]
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(
        logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1), y_train)
    p_cal = platt.predict_proba(logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]
    thr = _thresholds_from_oof(y_train, p_cal)
    return dict(name=name, model=search.best_estimator_, platt=platt, thresholds=thr,
                best_params=search.best_params_, oof_pr_auc=average_precision_score(y_train, p_cal),
                oof_roc_auc=roc_auc_score(y_train, p_cal))


print("WORKING... fitting the final models on all of TRAIN. Please wait (about 5-20 min).", flush=True)

ssp_pipe = build_model_pipeline(StabilityLR(random_state=SEED),
                                NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
ssp_params = {"model__threshold": [0.5, 0.6, 0.7, 0.8], "model__C_select": loguniform(5e-3, 5e-1),
              "model__C_final": loguniform(1e-2, 1e2), "model__class_weight": [None, "balanced"],
              "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))]}
final_ssp = fit_final("SSP-LR (final model)", ssp_pipe, ssp_params)

lr_pipe_f = build_model_pipeline(LogisticRegression(solver="liblinear", max_iter=5000),
                                 NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
lr_params_f = {"model__C": loguniform(1e-3, 1e2), "model__penalty": ["l1", "l2"],
               "model__class_weight": [None, "balanced"],
               "sampler": ["passthrough", smote_sampler(len(NUMERIC_MAIN))]}
final_lr = fit_final("Logistic Regression (comparator)", lr_pipe_f, lr_params_f)

joblib.dump({"ssp": final_ssp, "lr": final_lr}, OUT_DIR / "step07_final_models.joblib")
print("DONE - final models fitted and saved.\n", flush=True)

# --------------------------- which risk factors did SSP-LR keep? ---------------------------
m = final_ssp["model"].named_steps["model"]
names = encoded_feature_names(final_ssp["model"], NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN)
assert len(names) == m.n_features_in_, "feature-name mismatch - tell Claude"
coef = np.zeros(len(names)); coef[m.selected_] = m.final_.coef_[0]
feat = pd.DataFrame({"feature": names, "selection_freq": m.selection_freq_, "kept": m.selected_,
                     "odds_ratio": np.exp(coef)})
feat.loc[~feat.kept, "odds_ratio"] = np.nan
feat = feat.sort_values(["kept", "selection_freq"], ascending=[False, False]).reset_index(drop=True)
feat.to_csv(OUT_DIR / "step07_SSPLR_selected_features.csv", index=False)

print("=" * 70)
print("STEP 7.1 SUMMARY - FINAL MODEL FITTED ON TRAIN (TEST still locked)")
print("=" * 70)
p = final_ssp["best_params"]
print(f"\n[A] SSP-LR settings chosen: stability threshold {p['model__threshold']}, "
      f"C_select {p['model__C_select']:.4f}, C_final {p['model__C_final']:.3f}, "
      f"class_weight {p['model__class_weight']}, SMOTE {'yes' if p['sampler'] != 'passthrough' else 'no'}")
print(f"    Thresholds (from TRAIN out-of-fold): Youden {final_ssp['thresholds']['youden']:.4f}, "
      f"max-F2 {final_ssp['thresholds']['f2']:.4f}")
print(f"    TRAIN out-of-fold check: ROC-AUC {final_ssp['oof_roc_auc']:.3f}, PR-AUC {final_ssp['oof_pr_auc']:.3f}")

kept = feat[feat.kept]
print(f"\n[B] RISK FACTORS KEPT: {len(kept)} of {len(feat)} model columns")
print("    (stability = % of the 30 half-samples in which LASSO kept it;")
print("     odds ratio: numbers = per 1 SD increase, yes/no & groups = vs. not having it)")
for _, r in kept.iterrows():
    direction = "raises risk" if r.odds_ratio > 1 else "lowers risk"
    print(f"  {r.feature:<40} stability {100*r.selection_freq:5.1f}%   OR {r.odds_ratio:5.2f}  ({direction})")

dropped = feat[~feat.kept]
print(f"\n[C] DROPPED as not stable enough: {len(dropped)} columns")
print("   ", ", ".join(dropped.feature.tolist()))

q = final_lr["best_params"]
print(f"\n[D] Comparator LR: penalty {q['model__penalty']}, C {q['model__C']:.4f}, "
      f"class_weight {q['model__class_weight']}; uses all {len(names)} columns")
print("\nSaved: out/step07_final_models.joblib, out/step07_SSPLR_selected_features.csv")
print("=" * 70)

WORKING... fitting the final models on all of TRAIN. Please wait (about 5-20 min).
DONE - final models fitted and saved.

STEP 7.1 SUMMARY - FINAL MODEL FITTED ON TRAIN (TEST still locked)

[A] SSP-LR settings chosen: stability threshold 0.6, C_select 0.0669, C_final 0.797, class_weight balanced, SMOTE no
    Thresholds (from TRAIN out-of-fold): Youden 0.0309, max-F2 0.0594
    TRAIN out-of-fold check: ROC-AUC 0.844, PR-AUC 0.180

[B] RISK FACTORS KEPT: 32 of 52 model columns
    (stability = % of the 30 half-samples in which LASSO kept it;
     odds ratio: numbers = per 1 SD increase, yes/no & groups = vs. not having it)
  age                                      stability 100.0%   OR  2.16  (raises risk)
  income_ratio                             stability 100.0%   OR  0.83  (lowers risk)
  total_chol                               stability 100.0%   OR  0.91  (lowers risk)
  egfr                                     stability 100.0%   OR  0.79  (lowers risk)
  uacr                    

In [18]:
# ============================================================
# STEP 7.2  FINAL EVALUATION ON THE LOCKED TEST SET  (run ONCE)
#
# The TEST part (6,599 people, 265 strokes) has not been used for anything so far.
# The models, calibration and thresholds were all fixed on TRAIN in Step 7.1.
# Here they are applied UNCHANGED to TEST. Nothing is tuned or re-chosen after this.
# 95% confidence intervals: 2,000 bootstrap resamples of the TEST set.
# Needs: Cell 3.1, 4.0, 4.1 and 7.1 run in this session (7.1 defines StabilityLR).
# ============================================================
import joblib

assert "StabilityLR" in globals(), "Run Cell 7.1 first (it defines the SSP-LR model class)."
assert sha16(TEST_CSV) == "8a5d289373fe4a35", "TEST file changed! Stop and tell Claude."

print("WORKING... evaluating on TEST with bootstrap confidence intervals (about 1-3 min).", flush=True)

test = pd.read_csv(TEST_CSV)
X_test, y_test = test[FEATURES_MAIN], test[TARGET].values
final = joblib.load(OUT_DIR / "step07_final_models.joblib")

# v2 FIX (technical, no result had been computed yet):
# The 'was missing' flags were created only for columns that had missing values in TRAIN.
# If a TEST person is missing a value in a column that was never missing in TRAIN, scikit-learn
# stops with an error by default. Correct behaviour: the value is simply filled in (TRAIN median /
# most frequent value) and no new flag column is invented. This changes nothing that was learned.
for key in ["ssp", "lr"]:
    final[key]["model"].named_steps["fill"].named_transformers_["flags"].error_on_new = False
train_cols_missing = set(c for c in NUMERIC_MAIN + BINARY_MAIN if pd.read_csv(TRAIN_CSV, usecols=[c])[c].isna().any())
new_missing = [c for c in NUMERIC_MAIN + BINARY_MAIN if X_test[c].isna().any() and c not in train_cols_missing]
print(f"Columns missing in TEST but never missing in TRAIN: {new_missing if new_missing else 'none'} "
      f"({sum(int(X_test[c].isna().sum()) for c in new_missing)} values -> filled with TRAIN values)")


def calibrated_proba(f, X):
    p = f["model"].predict_proba(X)[:, 1]
    return f["platt"].predict_proba(logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]


def all_metrics(y, p, thr):
    slope, intercept = _calibration_slope_intercept(y, p)
    out = dict(roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
               brier=brier_score_loss(y, p), cal_slope=slope, cal_intercept=intercept)
    for tname, t in thr.items():
        for k, v in _metrics_at(y, p, t).items():
            out[f"{tname}_{k}"] = v
    return out


rng = np.random.RandomState(SEED)
B = 2000
boot_idx = [rng.choice(len(y_test), len(y_test), replace=True) for _ in range(B)]
boot_idx = [i for i in boot_idx if y_test[i].sum() > 0]          # every resample needs strokes

results, preds = {}, {}
for key in ["ssp", "lr"]:
    f = final[key]
    p = calibrated_proba(f, X_test)
    preds[key] = p
    thr = {"youden": f["thresholds"]["youden"], "f2": f["thresholds"]["f2"]}
    point = all_metrics(y_test, p, thr)
    boots = pd.DataFrame([all_metrics(y_test[i], p[i], thr) for i in boot_idx])
    ci = {k: (np.percentile(boots[k], 2.5), np.percentile(boots[k], 97.5)) for k in point
          if k not in ("youden_tp", "youden_fn", "youden_fp", "youden_tn",
                       "f2_tp", "f2_fn", "f2_fp", "f2_tn")}
    results[key] = dict(name=f["name"], point=point, ci=ci, boots=boots)

# paired difference SSP-LR minus LR (same bootstrap resamples)
d_auc = results["ssp"]["boots"]["roc_auc"] - results["lr"]["boots"]["roc_auc"]
d_pr = results["ssp"]["boots"]["pr_auc"] - results["lr"]["boots"]["pr_auc"]

# baseline on TEST
base = _metrics_at(y_test, np.zeros(len(y_test)), 0.5)

# save
pd.DataFrame({"seqn": test["seqn"], "y": y_test, "p_ssplr": preds["ssp"], "p_lr": preds["lr"]}) \
  .to_csv(OUT_DIR / "step07_test_predictions.csv", index=False)
rows = []
for key in ["ssp", "lr"]:
    r = results[key]
    for k, v in r["point"].items():
        lo, hi = r["ci"].get(k, (np.nan, np.nan))
        rows.append(dict(model=r["name"], metric=k, value=v, ci_low=lo, ci_high=hi))
pd.DataFrame(rows).to_csv(OUT_DIR / "step07_test_results.csv", index=False)
print("DONE.\n", flush=True)

# ------------------------------------------------------------------ print
def line(r, k, label, pct=False):
    v, (lo, hi) = r["point"][k], r["ci"][k]
    if pct:
        return f"    {label:<24} {100*v:5.1f}%   (95% CI {100*lo:5.1f} - {100*hi:5.1f})"
    return f"    {label:<24} {v:6.3f}   (95% CI {lo:.3f} - {hi:.3f})"

n, k = len(y_test), int(y_test.sum())
print("=" * 70)
print("STEP 7.2 SUMMARY - FINAL RESULT ON THE LOCKED TEST SET")
print("=" * 70)
print(f"\nTEST: {n:,} people, {k} strokes ({100*k/n:.2f}%)")
print(f"\n[A] BASELINE (always 'no stroke'): accuracy {100*base['accuracy']:.1f}%, recall 0.0%, "
      f"strokes found 0 of {k}")

for key, tag in [("ssp", "[B]"), ("lr", "[C]")]:
    r = results[key]; pt = r["point"]
    print(f"\n{tag} {r['name']}")
    print(line(r, "roc_auc", "ROC-AUC"))
    print(line(r, "pr_auc", "PR-AUC") + f"   (no-skill {k/n:.3f})")
    print(line(r, "brier", "Brier score"))
    print(line(r, "cal_slope", "Calibration slope") + "   (ideal 1.0)")
    print(line(r, "cal_intercept", "Calibration intercept") + "   (ideal 0.0)")
    print("  At the main threshold (Youden, fixed on TRAIN):")
    print(line(r, "youden_recall", "Recall (strokes found)", True)
          + f"   = {pt['youden_tp']} of {k}")
    print(line(r, "youden_accuracy", "Accuracy", True))
    print(line(r, "youden_specificity", "Specificity", True))
    print(line(r, "youden_precision", "Precision", True))
    print(line(r, "youden_npv", "NPV", True))
    print(f"    {'F1 / F2':<24} {pt['youden_f1']:.3f} / {pt['youden_f2']:.3f}")
    print(f"  Alternative threshold (max-F2): recall {100*pt['f2_recall']:.1f}% "
          f"({pt['f2_tp']} of {k}), accuracy {100*pt['f2_accuracy']:.1f}%")

print("\n[D] SSP-LR vs LOGISTIC REGRESSION on TEST (paired bootstrap)")
print(f"    ROC-AUC difference {d_auc.mean():+.4f}  (95% CI {np.percentile(d_auc, 2.5):+.4f} to "
      f"{np.percentile(d_auc, 97.5):+.4f})")
print(f"    PR-AUC  difference {d_pr.mean():+.4f}  (95% CI {np.percentile(d_pr, 2.5):+.4f} to "
      f"{np.percentile(d_pr, 97.5):+.4f})")
print("    (a CI that includes 0 = no real difference)")
print("\nSaved: out/step07_test_results.csv, out/step07_test_predictions.csv")
print("=" * 70)

WORKING... evaluating on TEST with bootstrap confidence intervals (about 1-3 min).
Columns missing in TEST but never missing in TRAIN: ['heart_disease'] (1 values -> filled with TRAIN values)
DONE.

STEP 7.2 SUMMARY - FINAL RESULT ON THE LOCKED TEST SET

TEST: 6,599 people, 265 strokes (4.02%)

[A] BASELINE (always 'no stroke'): accuracy 96.0%, recall 0.0%, strokes found 0 of 265

[B] SSP-LR (final model)
    ROC-AUC                   0.855   (95% CI 0.835 - 0.874)
    PR-AUC                    0.200   (95% CI 0.165 - 0.248)   (no-skill 0.040)
    Brier score               0.035   (95% CI 0.031 - 0.039)
    Calibration slope         1.033   (95% CI 0.944 - 1.131)   (ideal 1.0)
    Calibration intercept    -0.018   (95% CI -0.155 - 0.109)   (ideal 0.0)
  At the main threshold (Youden, fixed on TRAIN):
    Recall (strokes found)    86.0%   (95% CI  81.8 -  89.7)   = 228 of 265
    Accuracy                  69.7%   (95% CI  68.6 -  70.7)
    Specificity               69.0%   (95% CI  67.9

In [19]:
# ============================================================
# STEP 8.1  XAI VALIDATION - do THREE different explanation methods agree?  (gap G3)
#
# Each method ranks the 22 original risk factors by importance:
#   M1  Survey-weighted logistic regression (TRAIN, complete cases): Wald chi-square per factor
#       - uses the NHANES sample weights + clusters -> population-level odds ratios
#   M2  Permutation importance of the FINAL SSP-LR (TEST): how much PR-AUC drops when the
#       factor is randomly shuffled (30 repeats)
#   M3  SHAP values of the FINAL SSP-LR (TEST): mean |contribution| to each person's risk.
#       For a logistic model SHAP is exact: coefficient x (value - TRAIN average)
#       (identical to shap.LinearExplainer); columns of the same factor are added together.
# Agreement between the three rankings: Spearman rho and Kendall tau.
# "Robust risk factor" = in the top 10 of ALL three methods.
# Nothing here changes the model (explanation only).
# Needs: Cells 3.1, 4.0, 4.1 and 7.1 run in this session.
# ============================================================
import joblib
import statsmodels.api as sm
from scipy.stats import spearmanr, kendalltau, chi2
from sklearn.inspection import permutation_importance

assert "StabilityLR" in globals() and "encoded_feature_names" in globals(), "Run Cell 7.1 first."
print("WORKING... computing 3 explanation methods (about 2-6 min).", flush=True)

final = joblib.load(OUT_DIR / "step07_final_models.joblib")
pipe = final["ssp"]["model"]
pipe.named_steps["fill"].named_transformers_["flags"].error_on_new = False
test = pd.read_csv(TEST_CSV)
X_test, y_test = test[FEATURES_MAIN], test[TARGET].values

FACTORS = FEATURES_MAIN                                   # the 22 original risk factors


def encode(X):
    """Raw features -> the exact columns the model sees (no sampler at prediction time)."""
    Xc = pipe.named_steps["clean"].transform(X)
    Xf = pipe.named_steps["fill"].transform(Xc)
    return pipe.named_steps["encode"].transform(Xf)


enc_names = encoded_feature_names(pipe, NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN)
owner = [n.split(" (was missing)")[0].split(" = ")[0] for n in enc_names]   # column -> factor

# ---------------- M3  SHAP (exact for a logistic model) ----------------------------------
m = pipe.named_steps["model"]
coef_full = np.zeros(len(enc_names)); coef_full[m.selected_] = m.final_.coef_[0]
mu_train = encode(X_train).mean(axis=0)
shap_cols = (encode(X_test) - mu_train) * coef_full                    # people x columns
shap_factor = pd.DataFrame(shap_cols, columns=enc_names).T.groupby(owner).sum().T
M3 = shap_factor.abs().mean().reindex(FACTORS).fillna(0)

# ---------------- M2  Permutation importance (PR-AUC drop on TEST) -----------------------
perm = permutation_importance(pipe, X_test, y_test, scoring="average_precision",
                              n_repeats=30, random_state=SEED, n_jobs=N_JOBS)
M2 = pd.Series(perm.importances_mean, index=FACTORS)

# ---------------- M1  Survey-weighted logistic regression (TRAIN) ------------------------
d = train[FEATURES_MAIN + [TARGET, "weight_mec_6cyc", "psu", "strata"]].copy()
d = d[fixed_cleaning(d[FEATURES_MAIN]).notna().all(axis=1) & (d["weight_mec_6cyc"] > 0)]
Xs = fixed_cleaning(d[FEATURES_MAIN])
cols, groups = [], {}
for c in NUMERIC_MAIN:                                             # per 1 SD
    Xs[c] = (Xs[c] - Xs[c].mean()) / Xs[c].std(); cols.append(c); groups[c] = [c]
for c in BINARY_MAIN:
    cols.append(c); groups[c] = [c]
dummies = []
for c in CATEGORICAL_MAIN:                                         # reference = most common group
    ref = Xs[c].value_counts().idxmax()
    dm = pd.get_dummies(Xs[c], prefix=c, prefix_sep=" = ").drop(columns=f"{c} = {ref}").astype(float)
    dummies.append(dm); groups[c] = list(dm.columns)
Xd = pd.concat([Xs[NUMERIC_MAIN + BINARY_MAIN].astype(float)] + dummies, axis=1)
Xd = sm.add_constant(Xd)
w = d["weight_mec_6cyc"] / d["weight_mec_6cyc"].mean()
clusters = (d["strata"].astype(int) * 10 + d["psu"].astype(int)).values
glm = sm.GLM(d[TARGET].values, Xd, family=sm.families.Binomial(), var_weights=w.values)
res = glm.fit(cov_type="cluster", cov_kwds={"groups": clusters})
wald, pvals = {}, {}
for f, cs in groups.items():
    idx = [list(Xd.columns).index(c) for c in cs]
    b = res.params.values[idx]; V = res.cov_params().values[np.ix_(idx, idx)]
    stat = float(b @ np.linalg.pinv(V) @ b)
    wald[f], pvals[f] = stat - len(idx), chi2.sf(stat, len(idx))   # chi2 minus df (Harrell)
M1 = pd.Series(wald).reindex(FACTORS)
or_tab = pd.DataFrame({"OR": np.exp(res.params), "lo": np.exp(res.conf_int()[0]),
                       "hi": np.exp(res.conf_int()[1]), "p": res.pvalues}).drop(index="const")

# ---------------- agreement + robust factors ---------------------------------------------
imp = pd.DataFrame({"M1_survey_wald": M1, "M2_permutation": M2, "M3_shap": M3})
ranks = imp.rank(ascending=False, method="min")
imp = imp.join(ranks.add_suffix("_rank"))
TOP = 10
imp["robust"] = (ranks <= TOP).all(axis=1)
imp["survey_p"] = pd.Series(pvals)
imp.to_csv(OUT_DIR / "step08_xai_importance.csv")
or_tab.to_csv(OUT_DIR / "step08_survey_weighted_OR.csv")

pairs = [("M1_survey_wald", "M2_permutation"), ("M1_survey_wald", "M3_shap"), ("M2_permutation", "M3_shap")]
agree = {(a, b): (spearmanr(imp[a], imp[b])[0], kendalltau(imp[a], imp[b])[0]) for a, b in pairs}
print("DONE.\n", flush=True)

# ---------------- print ------------------------------------------------------------------
label = {"M1_survey_wald": "Survey OR (M1)", "M2_permutation": "Permutation (M2)", "M3_shap": "SHAP (M3)"}
print("=" * 72)
print("STEP 8.1 SUMMARY - DO 3 EXPLANATION METHODS AGREE?")
print("=" * 72)
print(f"M1 on TRAIN complete cases: {len(d):,} people ({int(d[TARGET].sum())} strokes), survey-weighted")
print(f"M2, M3 on TEST: {len(y_test):,} people (final SSP-LR model, unchanged)")

print("\n[A] AGREEMENT BETWEEN METHODS (1.0 = identical ranking, 0 = unrelated)")
for (a, b), (rho, tau) in agree.items():
    print(f"  {label[a]:<17} vs {label[b]:<17} Spearman rho {rho:5.2f} | Kendall tau {tau:5.2f}")

print(f"\n[B] RANK OF EACH RISK FACTOR (1 = most important)   * = ROBUST (top {TOP} in all 3)")
for f, r in imp.sort_values("M3_shap_rank").iterrows():
    star = "*" if r["robust"] else " "
    print(f" {star} {f:<15} survey-OR #{int(r['M1_survey_wald_rank']):>2} (p={r['survey_p']:.3f}) | "
          f"permutation #{int(r['M2_permutation_rank']):>2} | SHAP #{int(r['M3_shap_rank']):>2}")
robust = imp[imp.robust].index.tolist()
print(f"\n  ROBUST RISK FACTORS ({len(robust)}): {', '.join(robust)}")

print("\n[C] SURVEY-WEIGHTED ODDS RATIOS (population level; numbers per 1 SD, yes/no vs no)")
for c, r in or_tab.iterrows():
    sig = "significant" if r["p"] < 0.05 else "not significant"
    print(f"  {c:<34} OR {r['OR']:5.2f} (95% CI {r['lo']:.2f}-{r['hi']:.2f})  p={r['p']:.3f}  {sig}")
print("\nSaved: out/step08_xai_importance.csv, out/step08_survey_weighted_OR.csv")
print("=" * 72)

WORKING... computing 3 explanation methods (about 2-6 min).
DONE.

STEP 8.1 SUMMARY - DO 3 EXPLANATION METHODS AGREE?
M1 on TRAIN complete cases: 19,999 people (695 strokes), survey-weighted
M2, M3 on TEST: 6,599 people (final SSP-LR model, unchanged)

[A] AGREEMENT BETWEEN METHODS (1.0 = identical ranking, 0 = unrelated)
  Survey OR (M1)    vs Permutation (M2)  Spearman rho  0.84 | Kendall tau  0.68
  Survey OR (M1)    vs SHAP (M3)         Spearman rho  0.77 | Kendall tau  0.59
  Permutation (M2)  vs SHAP (M3)         Spearman rho  0.80 | Kendall tau  0.62

[B] RANK OF EACH RISK FACTOR (1 = most important)   * = ROBUST (top 10 in all 3)
 * age             survey-OR # 2 (p=0.000) | permutation # 1 | SHAP # 1
 * hypertension    survey-OR # 3 (p=0.000) | permutation # 3 | SHAP # 2
 * smoking         survey-OR # 4 (p=0.000) | permutation # 5 | SHAP # 3
 * income_ratio    survey-OR # 5 (p=0.000) | permutation #10 | SHAP # 4
 * egfr            survey-OR # 6 (p=0.000) | permutation # 4 | SHA

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/genmod/generalized_linear_model.py:1655: SpecificationWarning: cov_type not fully supported with var_weights
  warnings.warn('cov_type not fully supported with var_weights',


In [20]:
# ============================================================
# STEP 8.2  SURVEY-WEIGHTED ODDS RATIOS WITH EXACT DESIGN-BASED STANDARD ERRORS
#
# Why: Cell 8.1 printed a statsmodels warning ("cov_type not fully supported with
# var_weights"), so its confidence intervals were only approximate. Here the standard errors
# are computed the way NHANES recommends (Taylor linearisation, the same method as R 'survey'
# svyglm / SAS SURVEYLOGISTIC):
#   - weighted logistic regression (same estimates as 8.1),
#   - variance from the sampling design: PSUs (clusters) within strata,
#   - t-distribution with design degrees of freedom = #PSUs - #strata.
# Then the 3-method agreement and the robust-factor list are re-computed with these values.
# Needs: Cell 8.1 run in this session (re-uses Xd, d, groups, res, imp).
# ============================================================
from scipy.special import expit
from scipy.stats import t as t_dist, f as f_dist, spearmanr, kendalltau

assert "Xd" in globals() and "imp" in globals(), "Run Cell 8.1 first."

X = Xd.values.astype(float)
yv = d[TARGET].values.astype(float)
wv = d["weight_mec_6cyc"].values.astype(float)
strata = d["strata"].astype(int).values
psu = d["psu"].astype(int).values
beta = res.params.values                                   # weighted MLE (identical to 8.1)

p = expit(X @ beta)
A = (X * (wv * p * (1 - p))[:, None]).T @ X                # "bread"
U = X * (wv * (yv - p))[:, None]                          # each person's weighted score
meat = np.zeros((X.shape[1], X.shape[1]))
n_strata, n_psu, skipped = 0, 0, 0
for h in np.unique(strata):
    in_h = strata == h
    psus = np.unique(psu[in_h])
    if len(psus) < 2:
        skipped += 1; continue
    Uh = np.array([U[in_h & (psu == j)].sum(axis=0) for j in psus])   # PSU totals
    Uc = Uh - Uh.mean(axis=0)
    meat += len(psus) / (len(psus) - 1) * Uc.T @ Uc
    n_strata += 1; n_psu += len(psus)
Ainv = np.linalg.pinv(A)
V = Ainv @ meat @ Ainv
se = np.sqrt(np.diag(V))
df_design = n_psu - n_strata
tcrit = t_dist.ppf(0.975, df_design)

names_x = list(Xd.columns)
or_exact = pd.DataFrame({"OR": np.exp(beta), "lo": np.exp(beta - tcrit * se),
                         "hi": np.exp(beta + tcrit * se),
                         "p": 2 * t_dist.sf(np.abs(beta / se), df_design)}, index=names_x).drop(index="const")

wald2, p2 = {}, {}
for f, cs in groups.items():
    idx = [names_x.index(c) for c in cs]
    b = beta[idx]; Vf = V[np.ix_(idx, idx)]
    stat = float(b @ np.linalg.pinv(Vf) @ b)
    wald2[f] = stat - len(idx)
    p2[f] = f_dist.sf(stat / len(idx), len(idx), df_design)   # design-based F test

imp2 = imp[["M2_permutation", "M3_shap"]].copy()
imp2["M1_survey_wald"] = pd.Series(wald2).reindex(imp2.index)
ranks2 = imp2.rank(ascending=False, method="min")
imp2 = imp2.join(ranks2.add_suffix("_rank"))
imp2["robust"] = (ranks2 <= 10).all(axis=1)
imp2["survey_p"] = pd.Series(p2)
imp2.to_csv(OUT_DIR / "step08_xai_importance_exact.csv")
or_exact.to_csv(OUT_DIR / "step08_survey_weighted_OR_exact.csv")

# ---------------- print ------------------------------------------------------------------
print("=" * 72)
print("STEP 8.2 SUMMARY - SURVEY ODDS RATIOS WITH DESIGN-BASED (EXACT) ERRORS")
print("=" * 72)
print(f"Design: {n_strata} strata, {n_psu} PSUs, design df = {df_design}"
      + (f", {skipped} strata skipped (only 1 PSU)" if skipped else ""))

print("\n[A] ODDS RATIOS (numbers per 1 SD; yes/no and groups vs reference)")
for c, r in or_exact.iterrows():
    sig = "significant" if r["p"] < 0.05 else "not significant"
    print(f"  {c:<36} OR {r['OR']:5.2f} (95% CI {r['lo']:.2f}-{r['hi']:.2f})  p={r['p']:.3f}  {sig}")

print("\n[B] AGREEMENT BETWEEN METHODS (with exact survey statistics)")
for a, b in [("M1_survey_wald", "M2_permutation"), ("M1_survey_wald", "M3_shap"),
             ("M2_permutation", "M3_shap")]:
    print(f"  {a:<16} vs {b:<16} Spearman rho {spearmanr(imp2[a], imp2[b])[0]:5.2f} | "
          f"Kendall tau {kendalltau(imp2[a], imp2[b])[0]:5.2f}")

robust2 = imp2[imp2.robust].index.tolist()
print(f"\n[C] ROBUST RISK FACTORS (top 10 in all 3 methods): {len(robust2)}")
print("   ", ", ".join(robust2))
changed = sorted(set(robust2) ^ set(imp[imp.robust].index))
print(f"    Change vs Cell 8.1: {'none' if not changed else changed}")
print("\nSaved: out/step08_survey_weighted_OR_exact.csv, out/step08_xai_importance_exact.csv")
print("=" * 72)

STEP 8.2 SUMMARY - SURVEY ODDS RATIOS WITH DESIGN-BASED (EXACT) ERRORS
Design: 90 strata, 184 PSUs, design df = 94

[A] ODDS RATIOS (numbers per 1 SD; yes/no and groups vs reference)
  age                                  OR  2.02 (95% CI 1.70-2.41)  p=0.000  significant
  income_ratio                         OR  0.70 (95% CI 0.60-0.81)  p=0.000  significant
  sbp                                  OR  0.92 (95% CI 0.81-1.05)  p=0.208  not significant
  dbp                                  OR  1.07 (95% CI 0.97-1.19)  p=0.187  not significant
  glucose                              OR  1.01 (95% CI 0.88-1.16)  p=0.906  not significant
  hba1c                                OR  0.96 (95% CI 0.82-1.12)  p=0.577  not significant
  bmi                                  OR  1.01 (95% CI 0.76-1.34)  p=0.932  not significant
  waist                                OR  1.00 (95% CI 0.74-1.35)  p=0.985  not significant
  total_chol                           OR  0.93 (95% CI 0.82-1.06)  p=0.294  not 

In [21]:
# ============================================================
# STEP 8.3  XAI STABILITY - are the rankings stable if the data change a little?
#
# 100 bootstrap resamples (people drawn with replacement):
#   M1 survey-weighted LR  -> resample TRAIN complete cases, refit, design-based Wald (as 8.2)
#   M2 permutation         -> resample TEST, permutation importance (5 repeats each)
#   M3 SHAP                -> resample TEST, mean |SHAP| (exact linear SHAP from 8.1)
# For each factor we report the median rank and its 95% range in each method, and the
# % of resamples in which it is in the top 10 of ALL THREE methods ("robustness %").
# A factor with robustness >= 80% is called STABLY ROBUST.
# The model is NOT refitted - only the explanations are re-computed.
# Needs: Cells 8.1 and 8.2 run in this session.
# ============================================================
from scipy.special import expit
from sklearn.inspection import permutation_importance

for v in ["pipe", "X_test", "y_test", "shap_factor", "Xd", "yv", "wv", "strata", "psu", "groups", "names_x"]:
    assert v in globals(), f"'{v}' missing -> run Cells 8.1 and 8.2 first."

B, TOP = 100, 10
print(f"WORKING... {B} bootstrap resamples x 3 methods. Please wait (about 10-30 min). "
      "Progress is shown every 10 resamples.", flush=True)


def design_wald(Xm, y, w, st, ps):
    """Weighted logistic fit + Taylor-linearised design variance -> Wald (chi2 - df) per factor."""
    beta = sm.GLM(y, Xm, family=sm.families.Binomial(), var_weights=w).fit().params
    beta = np.asarray(beta)
    p = expit(Xm @ beta)
    A = (Xm * (w * p * (1 - p))[:, None]).T @ Xm
    U = Xm * (w * (y - p))[:, None]
    meat = np.zeros_like(A)
    for h in np.unique(st):
        in_h = st == h
        ps_h = np.unique(ps[in_h])
        if len(ps_h) < 2:
            continue
        Uh = np.array([U[in_h & (ps == j)].sum(axis=0) for j in ps_h])
        Uc = Uh - Uh.mean(axis=0)
        meat += len(ps_h) / (len(ps_h) - 1) * Uc.T @ Uc
    Ai = np.linalg.pinv(A)
    V = Ai @ meat @ Ai
    out = {}
    for f, cs in groups.items():
        idx = [names_x.index(c) for c in cs]
        b = beta[idx]
        out[f] = float(b @ np.linalg.pinv(V[np.ix_(idx, idx)]) @ b) - len(idx)
    return pd.Series(out)


Xmat = Xd.values.astype(float)
rng = np.random.RandomState(SEED)
rank_store = {"M1": [], "M2": [], "M3": []}
robust_hits = pd.Series(0, index=FACTORS, dtype=float)
t0 = time.time()
for b in range(1, B + 1):
    i_tr = rng.choice(len(yv), len(yv), replace=True)
    i_te = rng.choice(len(y_test), len(y_test), replace=True)
    m1 = design_wald(Xmat[i_tr], yv[i_tr], wv[i_tr], strata[i_tr], psu[i_tr]).reindex(FACTORS)
    m2 = pd.Series(permutation_importance(pipe, X_test.iloc[i_te], y_test[i_te],
                                          scoring="average_precision", n_repeats=5,
                                          random_state=SEED + b, n_jobs=N_JOBS).importances_mean,
                   index=FACTORS)
    m3 = shap_factor.iloc[i_te].abs().mean().reindex(FACTORS).fillna(0)
    r = pd.DataFrame({"M1": m1, "M2": m2, "M3": m3}).rank(ascending=False, method="min")
    for k in rank_store:
        rank_store[k].append(r[k])
    robust_hits += (r <= TOP).all(axis=1).astype(float)
    if b % 10 == 0:
        print(f"  {b}/{B} resamples done ({time.time() - t0:.0f}s)", flush=True)

stab = pd.DataFrame(index=FACTORS)
for k, lab in [("M1", "survey"), ("M2", "perm"), ("M3", "shap")]:
    R = pd.concat(rank_store[k], axis=1)
    stab[f"{lab}_median_rank"] = R.median(axis=1)
    stab[f"{lab}_rank_lo"] = R.quantile(0.025, axis=1)
    stab[f"{lab}_rank_hi"] = R.quantile(0.975, axis=1)
stab["robust_pct"] = 100 * robust_hits / B
stab = stab.sort_values("robust_pct", ascending=False)
stab.to_csv(OUT_DIR / "step08_xai_stability.csv")
print("DONE.\n", flush=True)

# ---------------- print ------------------------------------------------------------------
print("=" * 72)
print(f"STEP 8.3 SUMMARY - STABILITY OF THE EXPLANATIONS ({B} bootstrap resamples)")
print("=" * 72)
print("Robustness % = how often the factor was in the top 10 of ALL 3 methods")
print("Ranks: median [95% range] in each method (1 = most important)\n")
for f, r in stab.iterrows():
    tag = "STABLY ROBUST" if r.robust_pct >= 80 else ("sometimes" if r.robust_pct >= 20 else "")
    print(f"  {f:<15} robustness {r.robust_pct:5.1f}%  | survey {r.survey_median_rank:4.1f} "
          f"[{r.survey_rank_lo:.0f}-{r.survey_rank_hi:.0f}] | perm {r.perm_median_rank:4.1f} "
          f"[{r.perm_rank_lo:.0f}-{r.perm_rank_hi:.0f}] | SHAP {r.shap_median_rank:4.1f} "
          f"[{r.shap_rank_lo:.0f}-{r.shap_rank_hi:.0f}]  {tag}")
stable = stab[stab.robust_pct >= 80].index.tolist()
print(f"\nSTABLY ROBUST RISK FACTORS (>= 80% of resamples): {len(stable)}")
print("   ", ", ".join(stable))
print("\nSaved: out/step08_xai_stability.csv")
print("=" * 72)

WORKING... 100 bootstrap resamples x 3 methods. Please wait (about 10-30 min). Progress is shown every 10 resamples.
  10/100 resamples done (9s)
  20/100 resamples done (15s)
  30/100 resamples done (22s)
  40/100 resamples done (27s)
  50/100 resamples done (33s)
  60/100 resamples done (39s)
  70/100 resamples done (44s)
  80/100 resamples done (49s)
  90/100 resamples done (55s)
  100/100 resamples done (60s)
DONE.

STEP 8.3 SUMMARY - STABILITY OF THE EXPLANATIONS (100 bootstrap resamples)
Robustness % = how often the factor was in the top 10 of ALL 3 methods
Ranks: median [95% range] in each method (1 = most important)

  age             robustness 100.0%  | survey  2.0 [1-3] | perm  1.0 [1-1] | SHAP  1.0 [1-1]  STABLY ROBUST
  heart_disease   robustness 100.0%  | survey  1.5 [1-4] | perm  2.0 [2-4] | SHAP  6.0 [4-6]  STABLY ROBUST
  hypertension    robustness 100.0%  | survey  3.5 [2-7] | perm  3.0 [2-4] | SHAP  2.0 [2-2]  STABLY ROBUST
  egfr            robustness  80.0%  | surv

In [22]:
# ============================================================
# STEP 9.1  PREPARE THE KAGGLE DATASET FOR EXTERNAL VALIDATION  (runs in seconds)
#
# Kaggle "Stroke Prediction Dataset" (5,110 rows) - used ONLY as an external test.
# Nothing from Kaggle is used for training or tuning.
# Same cohort rules as the pilot (fixed in the plan): adults >= 18, gender "Other" removed.
# Kaggle columns are renamed / recoded to the NHANES "core" feature names:
#   age -> age | gender -> female | hypertension | heart_disease | avg_glucose_level -> glucose
#   bmi ("N/A" -> missing) | smoking_status -> smoking (never/former/current; "Unknown" -> missing)
#   ever_married (Yes/No) -> ever_married (1/0)
# Kaggle's work_type and Residence_type have no NHANES equivalent -> not used.
# Needs: Cell 3.1 (and 4.0 for the NHANES TRAIN comparison) run in this session.
# ============================================================
KAGGLE_CSV = ROOT / "data_external" / "stroke_dataset_raw.csv"
assert KAGGLE_CSV.exists(), f"File not found: {KAGGLE_CSV}"

kg_raw = pd.read_csv(KAGGLE_CSV)
print(f"Kaggle file fingerprint: {sha16(KAGGLE_CSV)} | rows {len(kg_raw):,} | columns {list(kg_raw.columns)}")

flow = [("Kaggle file", len(kg_raw), int(kg_raw["stroke"].sum()))]
kg = kg_raw[kg_raw["gender"].isin(["Male", "Female"])]
flow.append(("Gender 'Other' removed", len(kg), int(kg["stroke"].sum())))
kg = kg[kg["age"] >= 18].copy()
flow.append(("Adults (age >= 18)", len(kg), int(kg["stroke"].sum())))

kaggle = pd.DataFrame({
    "age": kg["age"].astype(float),
    "female": (kg["gender"] == "Female").astype(int),
    "hypertension": kg["hypertension"].astype(float),
    "heart_disease": kg["heart_disease"].astype(float),
    "glucose": kg["avg_glucose_level"].astype(float),
    "bmi": pd.to_numeric(kg["bmi"], errors="coerce"),
    "smoking": kg["smoking_status"].map({"never smoked": "never", "formerly smoked": "former",
                                         "smokes": "current"}),          # "Unknown" -> missing
    "ever_married": kg["ever_married"].map({"Yes": 1.0, "No": 0.0}),
    TARGET: kg["stroke"].astype(int),
}).reset_index(drop=True)
kaggle.to_csv(ROOT / "data_external" / "kaggle_core_harmonised.csv", index=False)

print("\n" + "=" * 70)
print("STEP 9.1 SUMMARY - KAGGLE DATASET PREPARED (external test only)")
print("=" * 70)
print("\n[A] WHO IS INCLUDED")
for step, n, k in flow:
    print(f"  {step:<28} {n:>6,} people | {k:>4} strokes ({100*k/n:.2f}%)")

print("\n[B] MISSING VALUES IN KAGGLE (core features)")
for c in FEATURES_CORE:
    print(f"  {c:<14} missing {100*kaggle[c].isna().mean():5.1f}%")

print("\n[C] CASE-MIX: NHANES TRAIN vs KAGGLE (how different are the two populations?)")
nh = train
def show(label, a, b, pct=False):
    if pct:
        print(f"  {label:<26} NHANES {100*a:6.1f}%   |   Kaggle {100*b:6.1f}%")
    else:
        print(f"  {label:<26} NHANES {a:6.1f}    |   Kaggle {b:6.1f}")
show("Stroke rate", nh[TARGET].mean(), kaggle[TARGET].mean(), True)
show("Age (mean)", nh["age"].mean(), kaggle["age"].mean())
show("Female", nh["female"].mean(), kaggle["female"].mean(), True)
show("Hypertension", nh["hypertension"].mean(), kaggle["hypertension"].mean(), True)
show("Heart disease", nh["heart_disease"].mean(), kaggle["heart_disease"].mean(), True)
show("Glucose (mean, mg/dL)", nh["glucose"].mean(), kaggle["glucose"].mean())
show("BMI (mean)", nh["bmi"].mean(), kaggle["bmi"].mean())
show("Ever married", nh["ever_married"].mean(), kaggle["ever_married"].mean(), True)
for s in ["never", "former", "current"]:
    show(f"Smoking: {s}", (nh["smoking"] == s).mean(), (kaggle["smoking"] == s).mean(), True)
show("Smoking: missing/Unknown", nh["smoking"].isna().mean(), kaggle["smoking"].isna().mean(), True)

print("\n[D] STROKE vs NO STROKE INSIDE KAGGLE (same direction as NHANES?)")
s1, s0 = kaggle[kaggle[TARGET] == 1], kaggle[kaggle[TARGET] == 0]
for c in ["age", "glucose", "bmi"]:
    print(f"  {c:<14} average: stroke {s1[c].mean():6.1f} vs no stroke {s0[c].mean():6.1f}")
for c in ["hypertension", "heart_disease", "ever_married", "female"]:
    print(f"  {c:<14} % yes:   stroke {100*s1[c].mean():5.1f}% vs no stroke {100*s0[c].mean():5.1f}%")
print("\nSaved: data_external/kaggle_core_harmonised.csv")
print("=" * 70)

Kaggle file fingerprint: 644d473b05d27970 | rows 5,110 | columns ['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']

STEP 9.1 SUMMARY - KAGGLE DATASET PREPARED (external test only)

[A] WHO IS INCLUDED
  Kaggle file                   5,110 people |  249 strokes (4.87%)
  Gender 'Other' removed        5,109 people |  249 strokes (4.87%)
  Adults (age >= 18)            4,253 people |  247 strokes (5.81%)

[B] MISSING VALUES IN KAGGLE (core features)
  age            missing   0.0%
  glucose        missing   0.0%
  bmi            missing   4.3%
  female         missing   0.0%
  hypertension   missing   0.0%
  heart_disease  missing   0.0%
  ever_married   missing   0.0%
  smoking        missing  20.3%

[C] CASE-MIX: NHANES TRAIN vs KAGGLE (how different are the two populations?)
  Stroke rate                NHANES    4.0%   |   Kaggle    5.8%
  Age (mean)                 NHANES   5

In [23]:
# ============================================================
# STEP 9.2  EXTERNAL VALIDATION ON KAGGLE  (the "core" model, 8 shared features)
#
# 1. Fit the CORE models on NHANES TRAIN only (same recipe as Step 7.1):
#      - Core SSP-LR (our method)      - Core Logistic Regression (comparator)
#    Tuning, Platt calibration and the Youden threshold all come from NHANES TRAIN.
# 2. Apply them UNCHANGED to (a) NHANES TEST (internal reference) and (b) Kaggle (external).
# 3. Recalibration check on Kaggle (descriptive only): how much would the risk levels need
#    shifting for a new population? Ranking ability (AUC) is not affected by this.
# 4. Kaggle rows with KNOWN smoking status only (20% of Kaggle has smoking = "Unknown").
# 5. Do the risk factors point in the same direction in both datasets?
# 95% CIs: 1,000 bootstrap resamples.
# Needs: Cells 3.1, 4.0, 4.1, 7.1 and 9.1 run in this session.
# ============================================================
import joblib
from scipy.special import expit

for v in ["StabilityLR", "encoded_feature_names", "kaggle"]:
    assert v in globals(), f"'{v}' missing -> run Cells 7.1 and 9.1 first."
print("WORKING... fitting core models on NHANES TRAIN and validating (about 5-15 min).", flush=True)


def fit_on(X, y, pipe, params):
    cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    search = RandomizedSearchCV(pipe, params, n_iter=N_ITER, scoring=TUNING_METRIC, cv=cv,
                                random_state=SEED, n_jobs=N_JOBS, refit=True).fit(X, y)
    oof = cross_val_predict(clone(search.best_estimator_), X, y, cv=cv,
                            method="predict_proba", n_jobs=N_JOBS)[:, 1]
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1), y)
    p_cal = platt.predict_proba(logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]
    model = search.best_estimator_
    model.named_steps["fill"].named_transformers_["flags"].error_on_new = False
    return dict(model=model, platt=platt, thr=_thresholds_from_oof(y, p_cal)["youden"],
                params=search.best_params_)


def predict_cal(f, X):
    p = f["model"].predict_proba(X)[:, 1]
    return f["platt"].predict_proba(logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]


def evaluate(y, p, thr, B=1000):
    pt = dict(roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
              brier=brier_score_loss(y, p), **_metrics_at(y, p, thr))
    pt["cal_slope"], pt["cal_intercept"] = _calibration_slope_intercept(y, p)
    rng = np.random.RandomState(SEED)
    bs = []
    for _ in range(B):
        i = rng.choice(len(y), len(y), replace=True)
        if y[i].sum() == 0:
            continue
        m = _metrics_at(y[i], p[i], thr)
        bs.append((roc_auc_score(y[i], p[i]), average_precision_score(y[i], p[i]), m["recall"], m["accuracy"]))
    bs = np.array(bs)
    pt["ci"] = {k: (np.percentile(bs[:, j], 2.5), np.percentile(bs[:, j], 97.5))
                for j, k in enumerate(["roc_auc", "pr_auc", "recall", "accuracy"])}
    return pt


# ---------------- 1. fit core models on NHANES TRAIN -------------------------------------
Xc_train = train[FEATURES_CORE]
core_ssp = fit_on(Xc_train, y_train,
                  build_model_pipeline(StabilityLR(random_state=SEED), NUMERIC_CORE, BINARY_CORE,
                                       CATEGORICAL_CORE, scale=True),
                  {"model__threshold": [0.5, 0.6, 0.7, 0.8], "model__C_select": loguniform(5e-3, 5e-1),
                   "model__C_final": loguniform(1e-2, 1e2), "model__class_weight": [None, "balanced"],
                   "sampler": ["passthrough", smote_sampler(len(NUMERIC_CORE))]})
core_lr = fit_on(Xc_train, y_train,
                 build_model_pipeline(LogisticRegression(solver="liblinear", max_iter=5000), NUMERIC_CORE,
                                      BINARY_CORE, CATEGORICAL_CORE, scale=True),
                 {"model__C": loguniform(1e-3, 1e2), "model__penalty": ["l1", "l2"],
                  "model__class_weight": [None, "balanced"],
                  "sampler": ["passthrough", smote_sampler(len(NUMERIC_CORE))]})
joblib.dump({"core_ssp": core_ssp, "core_lr": core_lr}, OUT_DIR / "step09_core_models.joblib")

# ---------------- 2. evaluate: NHANES TEST and Kaggle ------------------------------------
test = pd.read_csv(TEST_CSV)
sets = {"NHANES TEST": (test[FEATURES_CORE], test[TARGET].values),
        "KAGGLE (external)": (kaggle[FEATURES_CORE], kaggle[TARGET].values)}
known = kaggle["smoking"].notna()
sets["KAGGLE, smoking known"] = (kaggle.loc[known, FEATURES_CORE], kaggle.loc[known, TARGET].values)

res9, kg_pred = {}, {}
for mname, f in [("Core SSP-LR", core_ssp), ("Core LR", core_lr)]:
    for sname, (X, y) in sets.items():
        p = predict_cal(f, X)
        res9[(mname, sname)] = evaluate(y, p, f["thr"])
        if sname == "KAGGLE (external)":
            kg_pred[mname] = p

# ---------------- 3. recalibration check on Kaggle (descriptive) -------------------------
yk = kaggle[TARGET].values
recal = {}
for mname in kg_pred:
    lp = logit(np.clip(kg_pred[mname], 1e-6, 1 - 1e-6))
    a = _calibration_slope_intercept(yk, kg_pred[mname])[1]              # intercept-only update
    p_int = expit(a + lp)
    lr2 = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), yk)   # slope + intercept
    p_log = lr2.predict_proba(lp.reshape(-1, 1))[:, 1]
    recal[mname] = dict(mean_pred=kg_pred[mname].mean(), obs=yk.mean(), shift=a,
                        brier_before=brier_score_loss(yk, kg_pred[mname]),
                        brier_int=brier_score_loss(yk, p_int), brier_log=brier_score_loss(yk, p_log))

# ---------------- 5. risk-factor directions: NHANES core LR vs LR fitted on Kaggle -------
m_nh = core_lr["model"]
names_c = encoded_feature_names(m_nh, NUMERIC_CORE, BINARY_CORE, CATEGORICAL_CORE)
or_nh = pd.Series(np.exp(m_nh.named_steps["model"].coef_[0]), index=names_c)
kg_fit = clone(m_nh).set_params(model__C=1e6, model__penalty="l2", model__class_weight=None,
                                sampler="passthrough").fit(kaggle[FEATURES_CORE], yk)
names_k = encoded_feature_names(kg_fit, NUMERIC_CORE, BINARY_CORE, CATEGORICAL_CORE)
or_kg = pd.Series(np.exp(kg_fit.named_steps["model"].coef_[0]), index=names_k)
dirs = pd.DataFrame({"NHANES_OR": or_nh, "Kaggle_OR": or_kg}).dropna()
dirs = dirs[~dirs.index.str.contains("missing")]
dirs["same_direction"] = np.sign(np.log(dirs.NHANES_OR)) == np.sign(np.log(dirs.Kaggle_OR))
dirs.to_csv(OUT_DIR / "step09_direction_check.csv")

rows = [dict(model=m, dataset=s, **{k: v for k, v in r.items() if k != "ci"},
             **{f"{k}_lo": v[0] for k, v in r["ci"].items()}, **{f"{k}_hi": v[1] for k, v in r["ci"].items()})
        for (m, s), r in res9.items()]
pd.DataFrame(rows).to_csv(OUT_DIR / "step09_external_validation.csv", index=False)
print("DONE.\n", flush=True)

# ---------------- print ------------------------------------------------------------------
print("=" * 72)
print("STEP 9.2 SUMMARY - EXTERNAL VALIDATION (core model, 8 shared features)")
print("=" * 72)
print(f"Core SSP-LR kept {int(core_ssp['model'].named_steps['model'].selected_.sum())} of "
      f"{core_ssp['model'].named_steps['model'].n_features_in_} columns | Youden threshold "
      f"{core_ssp['thr']:.4f} (from NHANES TRAIN)")
base_k = _metrics_at(yk, np.zeros(len(yk)), 0.5)
print(f"Kaggle baseline (always 'no stroke'): accuracy {100*base_k['accuracy']:.1f}%, recall 0%")

for mname in ["Core SSP-LR", "Core LR"]:
    print(f"\n[{mname}]")
    for sname in sets:
        r = res9[(mname, sname)]; ci = r["ci"]
        print(f"  {sname:<22} ROC-AUC {r['roc_auc']:.3f} ({ci['roc_auc'][0]:.3f}-{ci['roc_auc'][1]:.3f}) | "
              f"PR-AUC {r['pr_auc']:.3f} | recall {100*r['recall']:4.1f}% ({100*ci['recall'][0]:.0f}-"
              f"{100*ci['recall'][1]:.0f}) = {r['tp']} of {r['tp']+r['fn']} | accuracy {100*r['accuracy']:4.1f}% | "
              f"cal. slope {r['cal_slope']:.2f}, intercept {r['cal_intercept']:+.2f}")

print("\n[RECALIBRATION CHECK ON KAGGLE] (descriptive; AUC is unchanged by recalibration)")
for mname, r in recal.items():
    print(f"  {mname:<12} average predicted risk {100*r['mean_pred']:.1f}% vs observed {100*r['obs']:.1f}% | "
          f"needed log-odds shift {r['shift']:+.2f} | Brier {r['brier_before']:.4f} -> "
          f"{r['brier_int']:.4f} (intercept update) -> {r['brier_log']:.4f} (full recalibration)")

print("\n[RISK-FACTOR DIRECTIONS] NHANES core LR vs LR fitted on Kaggle (OR > 1 = raises risk)")
for c, r in dirs.iterrows():
    print(f"  {c:<22} NHANES OR {r.NHANES_OR:5.2f} | Kaggle OR {r.Kaggle_OR:5.2f} | "
          f"{'same direction' if r.same_direction else 'OPPOSITE'}")
print(f"  Same direction: {int(dirs.same_direction.sum())} of {len(dirs)}")
print("\nSaved: out/step09_external_validation.csv, out/step09_direction_check.csv, out/step09_core_models.joblib")
print("=" * 72)

WORKING... fitting core models on NHANES TRAIN and validating (about 5-15 min).
DONE.

STEP 9.2 SUMMARY - EXTERNAL VALIDATION (core model, 8 shared features)
Core SSP-LR kept 9 of 15 columns | Youden threshold 0.0339 (from NHANES TRAIN)
Kaggle baseline (always 'no stroke'): accuracy 94.2%, recall 0%

[Core SSP-LR]
  NHANES TEST            ROC-AUC 0.839 (0.818-0.859) | PR-AUC 0.176 | recall 83.8% (79-88) = 222 of 265 | accuracy 68.6% | cal. slope 1.02, intercept -0.01
  KAGGLE (external)      ROC-AUC 0.819 (0.797-0.842) | PR-AUC 0.192 | recall 72.1% (66-78) = 178 of 247 | accuracy 76.7% | cal. slope 0.94, intercept +0.70
  KAGGLE, smoking known  ROC-AUC 0.810 (0.783-0.836) | PR-AUC 0.189 | recall 71.3% (65-78) = 144 of 202 | accuracy 75.6% | cal. slope 0.91, intercept +0.70

[Core LR]
  NHANES TEST            ROC-AUC 0.840 (0.819-0.860) | PR-AUC 0.176 | recall 84.5% (80-88) = 224 of 265 | accuracy 68.3% | cal. slope 1.02, intercept -0.01
  KAGGLE (external)      ROC-AUC 0.818 (0.797-0.8

In [24]:
# ============================================================
# STEP 10.1  SENSITIVITY ANALYSES (pre-declared in the analysis plan)
#
# Question: do our conclusions change if we make different reasonable choices?
#   S1  ADD depression score (PHQ-9) + trouble sleeping  (possible reverse causation)
#   S2  COMPLETE CASES only (no imputation at all)
#   S3  SURVEY-WEIGHTED evaluation of the final model on TEST (US-population performance)
#   S4  NO "was missing" flags (added BEFORE the test was opened; flags looked like
#       disability markers, e.g. "BMI could not be measured")
# S1, S2, S4 refit SSP-LR on NHANES TRAIN with exactly the same recipe as the final model
# (3-fold tuning, 30 candidates, Platt calibration, Youden threshold from TRAIN) and are
# evaluated on TEST. S3 re-uses the final model unchanged.
# Needs: Cells 3.1, 4.0, 4.1 and 7.1 run in this session.
# ============================================================
import joblib
from scipy.special import expit

assert "StabilityLR" in globals(), "Run Cell 7.1 first."
print("WORKING... 3 refits + 1 weighted evaluation. Please wait (about 15-40 min).", flush=True)

test = pd.read_csv(TEST_CSV)
y_test = test[TARGET].values
final = joblib.load(OUT_DIR / "step07_final_models.joblib")
main_pred = pd.read_csv(OUT_DIR / "step07_test_predictions.csv")
assert (main_pred["seqn"].values == test["seqn"].values).all()
p_main, thr_main = main_pred["p_ssplr"].values, final["ssp"]["thresholds"]["youden"]


def ssp_params(n_num):
    return {"model__threshold": [0.5, 0.6, 0.7, 0.8], "model__C_select": loguniform(5e-3, 5e-1),
            "model__C_final": loguniform(1e-2, 1e2), "model__class_weight": [None, "balanced"],
            "sampler": ["passthrough", smote_sampler(n_num)]}


def fit_variant(X, y, pipe, params):
    cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    s = RandomizedSearchCV(pipe, params, n_iter=N_ITER, scoring=TUNING_METRIC, cv=cv,
                           random_state=SEED, n_jobs=N_JOBS, refit=True).fit(X, y)
    oof = cross_val_predict(clone(s.best_estimator_), X, y, cv=cv, method="predict_proba", n_jobs=N_JOBS)[:, 1]
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1), y)
    p_oof = platt.predict_proba(logit(np.clip(oof, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]
    m = s.best_estimator_
    flags = m.named_steps["fill"].named_transformers_["flags"]
    if hasattr(flags, "error_on_new"):
        flags.error_on_new = False
    return dict(model=m, platt=platt, thr=_thresholds_from_oof(y, p_oof)["youden"])


def pred(f, X):
    p = f["model"].predict_proba(X)[:, 1]
    return f["platt"].predict_proba(logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]


def summary(y, p, thr, w=None):
    pr = (p >= thr).astype(int)
    w = np.ones(len(y)) if w is None else w
    tp = w[(pr == 1) & (y == 1)].sum(); fn = w[(pr == 0) & (y == 1)].sum()
    tn = w[(pr == 0) & (y == 0)].sum(); fp = w[(pr == 1) & (y == 0)].sum()
    slope, inter = _calibration_slope_intercept(y, p)
    return dict(roc_auc=roc_auc_score(y, p, sample_weight=w), pr_auc=average_precision_score(y, p, sample_weight=w),
                recall=tp / (tp + fn), accuracy=(tp + tn) / w.sum(), specificity=tn / (tn + fp),
                precision=tp / (tp + fp) if tp + fp else 0, cal_slope=slope, cal_intercept=inter,
                n=len(y), strokes=int(y.sum()))


def paired_auc_diff(y, pa, pb, B=1000):
    rng = np.random.RandomState(SEED); d = []
    for _ in range(B):
        i = rng.choice(len(y), len(y), replace=True)
        if y[i].sum():
            d.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    return np.mean(d), np.percentile(d, 2.5), np.percentile(d, 97.5)


out = {"MAIN (final SSP-LR)": summary(y_test, p_main, thr_main)}
diffs, extra = {}, {}

# ---- S1: add PHQ-9 + trouble sleeping ------------------------------------------------------
num1, bin1 = NUMERIC_MAIN + ["phq9"], BINARY_MAIN + ["sleep_trouble"]
f1 = fit_variant(train[num1 + bin1 + CATEGORICAL_MAIN], y_train,
                 build_model_pipeline(StabilityLR(random_state=SEED), num1, bin1, CATEGORICAL_MAIN, scale=True),
                 ssp_params(len(num1)))
p1 = pred(f1, test[num1 + bin1 + CATEGORICAL_MAIN])
out["S1 + PHQ-9 & sleep"] = summary(y_test, p1, f1["thr"])
diffs["S1 + PHQ-9 & sleep"] = paired_auc_diff(y_test, p1, p_main)
print("  S1 done", flush=True)

# ---- S2: complete cases only ---------------------------------------------------------------
cc_tr = fixed_cleaning(train[FEATURES_MAIN]).notna().all(axis=1).values
cc_te = fixed_cleaning(test[FEATURES_MAIN]).notna().all(axis=1).values
f2 = fit_variant(train.loc[cc_tr, FEATURES_MAIN], y_train[cc_tr],
                 build_model_pipeline(StabilityLR(random_state=SEED), NUMERIC_MAIN, BINARY_MAIN,
                                      CATEGORICAL_MAIN, scale=True), ssp_params(len(NUMERIC_MAIN)))
p2 = pred(f2, test.loc[cc_te, FEATURES_MAIN])
out["S2 complete cases"] = summary(y_test[cc_te], p2, f2["thr"])
out["   (MAIN on same complete-case TEST rows)"] = summary(y_test[cc_te], p_main[cc_te], thr_main)
print("  S2 done", flush=True)

# ---- S3: survey-weighted evaluation of the final model ---------------------------------------
w_te = test["weight_mec_6cyc"].values
out["S3 survey-weighted (MAIN)"] = summary(y_test, p_main, thr_main, w=w_te)
extra["S3"] = dict(weighted_prev=np.average(y_test, weights=w_te), weighted_pred=np.average(p_main, weights=w_te))
print("  S3 done", flush=True)

# ---- S4: no 'was missing' flags --------------------------------------------------------------
pipe4 = build_model_pipeline(StabilityLR(random_state=SEED), NUMERIC_MAIN, BINARY_MAIN, CATEGORICAL_MAIN, scale=True)
pipe4.set_params(fill__flags="drop")
f4 = fit_variant(X_train, y_train, pipe4, ssp_params(len(NUMERIC_MAIN)))
p4 = pred(f4, test[FEATURES_MAIN])
out["S4 no missing-flags"] = summary(y_test, p4, f4["thr"])
diffs["S4 no missing-flags"] = paired_auc_diff(y_test, p4, p_main)
m4 = f4["model"]
oh = m4.named_steps["encode"].named_transformers_["cat"]
ordc = m4.named_steps["fill"].named_transformers_["cat"].named_steps["ord"].categories_
names4 = list(NUMERIC_MAIN) + [f"{c} = {ordc[j][int(v)]}" for j, c in enumerate(CATEGORICAL_MAIN)
                               for v in oh.categories_[j]] + list(BINARY_MAIN)
mm = m4.named_steps["model"]
coef4 = np.zeros(len(names4)); coef4[mm.selected_] = mm.final_.coef_[0]
s4tab = pd.DataFrame({"feature": names4, "stability": mm.selection_freq_, "kept": mm.selected_,
                      "OR": np.exp(coef4)}).query("kept").sort_values("stability", ascending=False)
s4tab.to_csv(OUT_DIR / "step10_S4_kept_features.csv", index=False)
print("  S4 done", flush=True)

pd.DataFrame(out).T.to_csv(OUT_DIR / "step10_sensitivity.csv")
print("DONE.\n", flush=True)

# ---------------- print ------------------------------------------------------------------
print("=" * 74)
print("STEP 10.1 SUMMARY - SENSITIVITY ANALYSES (all evaluated on the TEST set)")
print("=" * 74)
for k, r in out.items():
    print(f"  {k:<42} n={r['n']:>5,} ({r['strokes']} strokes) | AUC {r['roc_auc']:.3f} | PR-AUC {r['pr_auc']:.3f} | "
          f"recall {100*r['recall']:4.1f}% | acc {100*r['accuracy']:4.1f}% | cal.slope {r['cal_slope']:.2f}")

print("\n  AUC difference vs MAIN on the same people (paired bootstrap, 95% CI; includes 0 = no change):")
for k, (m_, lo, hi) in diffs.items():
    print(f"    {k:<24} {m_:+.4f}  ({lo:+.4f} to {hi:+.4f})")

e = extra["S3"]
print(f"\n  S3: US-population (weighted) stroke prevalence {100*e['weighted_prev']:.2f}% | "
      f"weighted average predicted risk {100*e['weighted_pred']:.2f}%")

print(f"\n  S4: without missing-flags SSP-LR keeps {len(s4tab)} columns. Top risk factors (stability, OR):")
for _, r in s4tab.head(12).iterrows():
    print(f"    {r.feature:<34} {100*r.stability:5.1f}%  OR {r.OR:5.2f}")
print("\nSaved: out/step10_sensitivity.csv, out/step10_S4_kept_features.csv")
print("=" * 74)

WORKING... 3 refits + 1 weighted evaluation. Please wait (about 15-40 min).
  S1 done
  S2 done
  S3 done
  S4 done
DONE.

STEP 10.1 SUMMARY - SENSITIVITY ANALYSES (all evaluated on the TEST set)
  MAIN (final SSP-LR)                        n=6,599 (265 strokes) | AUC 0.855 | PR-AUC 0.200 | recall 86.0% | acc 69.7% | cal.slope 1.03
  S1 + PHQ-9 & sleep                         n=6,599 (265 strokes) | AUC 0.862 | PR-AUC 0.208 | recall 81.9% | acc 75.7% | cal.slope 1.04
  S2 complete cases                          n=4,981 (177 strokes) | AUC 0.862 | PR-AUC 0.197 | recall 89.3% | acc 68.6% | cal.slope 1.13
     (MAIN on same complete-case TEST rows)  n=4,981 (177 strokes) | AUC 0.861 | PR-AUC 0.194 | recall 85.3% | acc 71.7% | cal.slope 1.15
  S3 survey-weighted (MAIN)                  n=6,599 (265 strokes) | AUC 0.881 | PR-AUC 0.175 | recall 85.1% | acc 76.5% | cal.slope 1.03
  S4 no missing-flags                        n=6,599 (265 strokes) | AUC 0.855 | PR-AUC 0.188 | recall 85.7% | acc

In [25]:
# ============================================================
# STEP 11.1  THESIS FIGURES  -> saved in the folder  figures/
#
# Every figure is saved twice: PNG (300 dpi, for Word) and PDF (vector, for LaTeX).
# Each figure is made inside its own try-block: if one fails, the others are still made
# and the summary tells you which one failed.
# Only reads saved results (out/...), the TEST/Kaggle files and the saved models.
# Nothing is re-trained or re-tuned.
# Needs: Cells 3.1, 4.0, 4.1 and 7.1 run in this session (7.1 defines the SSP-LR class).
# ============================================================
import joblib
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
print(f"WORKING... creating figures in {FIG_DIR} (about 1-2 min).", flush=True)

# ---- one consistent, colour-blind-safe style for every figure ----
BLUE, ORANGE, AQUA, GREY, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#9a9890", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11,
    "axes.labelsize": 10, "axes.edgecolor": "#b5b3ad", "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0",
    "grid.linewidth": 0.6, "legend.frameon": False, "lines.linewidth": 2,
})
made, failed = [], []


def save(fig, name, caption):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    made.append((name, caption))


def cal_points(y, p, bins=10):
    q = pd.qcut(p, bins, labels=False, duplicates="drop")
    g = pd.DataFrame({"y": y, "p": p, "q": q}).groupby("q")
    return g["p"].mean().values, g["y"].mean().values, g.size().values


# ---- shared data ----
test = pd.read_csv(TEST_CSV)
y_te = test[TARGET].values
tp_ = pd.read_csv(OUT_DIR / "step07_test_predictions.csv")
p_ssp, p_lr = tp_["p_ssplr"].values, tp_["p_lr"].values
final = joblib.load(OUT_DIR / "step07_final_models.joblib")
thr_ssp = final["ssp"]["thresholds"]["youden"]
prev = y_te.mean()

# ---------------------------------------------------------------- Fig 01 flow
try:
    fl = pd.read_csv(OUT_DIR / "step02_flow.csv")
    fig, ax = plt.subplots(figsize=(6.2, 5.6)); ax.axis("off"); ax.grid(False)
    n = len(fl); ys = np.linspace(0.93, 0.33, n)
    for i, (_, r) in enumerate(fl.iterrows()):
        ax.text(0.5, ys[i], f"{r['step']}\nn = {r['remaining']:,}", ha="center", va="center",
                fontsize=9.5, bbox=dict(boxstyle="round,pad=0.5", fc="#f2f5f8", ec="#b5b3ad"))
        if i < n - 1:
            ax.annotate("", xy=(0.5, ys[i + 1] + 0.055), xytext=(0.5, ys[i] - 0.055),
                        arrowprops=dict(arrowstyle="->", color=MUTED))
            if fl.iloc[i + 1]["removed"]:
                ax.text(0.78, (ys[i] + ys[i + 1]) / 2, f"excluded {int(fl.iloc[i+1]['removed']):,}",
                        fontsize=8.5, color=MUTED, va="center")
    ax.text(0.27, 0.12, "TRAIN 80%\nn = 26,394 (1,057 strokes)", ha="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.5", fc="#e8f1fc", ec=BLUE))
    ax.text(0.73, 0.12, "TEST 20% (locked)\nn = 6,599 (265 strokes)", ha="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.5", fc="#fdeee7", ec=ORANGE))
    ax.annotate("", xy=(0.27, 0.19), xytext=(0.45, ys[-1] - 0.06), arrowprops=dict(arrowstyle="->", color=MUTED))
    ax.annotate("", xy=(0.73, 0.19), xytext=(0.55, ys[-1] - 0.06), arrowprops=dict(arrowstyle="->", color=MUTED))
    ax.set_title("Participant flow, NHANES 2007-2018", loc="left")
    save(fig, "Fig01_participant_flow", "Inclusion flow and the stratified 80/20 split")
except Exception as e:
    failed.append(("Fig01", repr(e)))

# ---------------------------------------------------------------- Fig 02 accuracy paradox
try:
    base = _metrics_at(y_te, np.zeros(len(y_te)), 0.5)
    ssp = _metrics_at(y_te, p_ssp, thr_ssp)
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)
    for ax, key, title in [(axes[0], "accuracy", "Accuracy"), (axes[1], "recall", "Recall (strokes found)")]:
        vals = [100 * base[key], 100 * ssp[key]]
        bars = ax.bar(["Always 'no stroke'", "SSP-LR (final)"], vals, color=[GREY, BLUE], width=0.55)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 2, f"{v:.1f}%", ha="center", color=INK, fontsize=10)
        ax.set_title(title, loc="left"); ax.set_ylim(0, 110); ax.grid(axis="x", visible=False)
    axes[0].set_ylabel("%")
    fig.suptitle(f"The accuracy paradox on the locked TEST set ({int(y_te.sum())} strokes of {len(y_te):,})",
                 x=0.02, ha="left", fontsize=11)
    save(fig, "Fig02_accuracy_paradox", "Baseline vs final model: accuracy vs recall")
except Exception as e:
    failed.append(("Fig02", repr(e)))

# ---------------------------------------------------------------- Fig 03 model comparison
try:
    cmp_ = pd.read_csv(OUT_DIR / "step06_model_comparison.csv").sort_values("pr_auc")
    fig, axes = plt.subplots(1, 2, figsize=(9, 5.2), sharey=True)
    ypos = np.arange(len(cmp_))
    colors = [ORANGE if v == "significantly worse" else (BLUE if "SSP" in m else MUTED)
              for v, m in zip(cmp_["verdict"], cmp_["model"])]
    axes[0].errorbar(cmp_["pr_auc"], ypos, xerr=cmp_["pr_auc_sd"], fmt="none", ecolor="#c9c7c0", lw=1.5)
    axes[0].scatter(cmp_["pr_auc"], ypos, c=colors, s=45, zorder=3)
    axes[0].axvline(prev, color=GREY, ls="--", lw=1); axes[0].text(prev, len(cmp_) - 0.4, " no-skill", color=MUTED, fontsize=8)
    axes[0].set_yticks(ypos); axes[0].set_yticklabels(cmp_["model"]); axes[0].set_xlabel("PR-AUC (mean ± SD, 15 folds)")
    axes[1].scatter(cmp_["roc_auc"], ypos, c=colors, s=45, zorder=3); axes[1].set_xlabel("ROC-AUC (mean, 15 folds)")
    for ax in axes:
        ax.grid(axis="y", visible=False)
    from matplotlib.lines import Line2D
    axes[1].legend(handles=[Line2D([], [], marker="o", ls="", color=BLUE, label="Final model (SSP-LR)"),
                            Line2D([], [], marker="o", ls="", color=MUTED, label="Equivalent to best"),
                            Line2D([], [], marker="o", ls="", color=ORANGE, label="Significantly worse")],
                   loc="lower right", fontsize=8.5)
    fig.suptitle("Nested cross-validation on TRAIN: 16 models", x=0.02, ha="left", fontsize=11)
    save(fig, "Fig03_model_comparison", "All models, nested CV, with statistical verdict")
except Exception as e:
    failed.append(("Fig03", repr(e)))

# ---------------------------------------------------------------- Fig 04 ROC + PR on TEST
try:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for p, lab, col, ls in [(p_ssp, "SSP-LR (final)", BLUE, "-"), (p_lr, "Logistic regression", ORANGE, "--")]:
        f_, t_, _ = roc_curve(y_te, p)
        axes[0].plot(f_, t_, color=col, ls=ls, label=f"{lab}, AUC {roc_auc_score(y_te, p):.3f}")
        pr_, rc_, _ = precision_recall_curve(y_te, p)
        axes[1].plot(rc_, pr_, color=col, ls=ls, label=f"{lab}, AP {average_precision_score(y_te, p):.3f}")
    axes[0].plot([0, 1], [0, 1], color=GREY, ls=":", lw=1, label="Chance")
    axes[1].axhline(prev, color=GREY, ls=":", lw=1, label=f"No-skill ({prev:.3f})")
    ssp_m = _metrics_at(y_te, p_ssp, thr_ssp)
    axes[0].scatter(1 - ssp_m["specificity"], ssp_m["recall"], color=BLUE, s=50, zorder=4, edgecolor="white")
    axes[0].annotate("Youden threshold\n(fixed on TRAIN)", (1 - ssp_m["specificity"], ssp_m["recall"]),
                     xytext=(0.45, 0.55), fontsize=8.5, color=MUTED, arrowprops=dict(arrowstyle="-", color=GREY))
    axes[0].set(xlabel="1 - specificity", ylabel="Recall (sensitivity)", title="ROC curve")
    axes[1].set(xlabel="Recall (sensitivity)", ylabel="Precision", title="Precision-recall curve")
    for ax in axes:
        ax.legend(fontsize=8.5, loc="lower right" if ax is axes[0] else "upper right")
    fig.suptitle(f"Locked TEST set (n = {len(y_te):,}, {int(y_te.sum())} strokes)", x=0.02, ha="left", fontsize=11)
    save(fig, "Fig04_ROC_PR_test", "ROC and PR curves of the final model and LR on TEST")
except Exception as e:
    failed.append(("Fig04", repr(e)))

# ---------------------------------------------------------------- Fig 05 calibration (TEST + Kaggle)
kg_p = None
try:
    kaggle_h = pd.read_csv(ROOT / "data_external" / "kaggle_core_harmonised.csv")
    core = joblib.load(OUT_DIR / "step09_core_models.joblib")["core_ssp"]
    core["model"].named_steps["fill"].named_transformers_["flags"].error_on_new = False
    raw = core["model"].predict_proba(kaggle_h[FEATURES_CORE])[:, 1]
    kg_p = core["platt"].predict_proba(logit(np.clip(raw, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]
    kg_y = kaggle_h[TARGET].values
except Exception as e:
    failed.append(("Kaggle predictions (Fig05b, Fig07)", repr(e)))
try:
    panels = [("NHANES TEST - final SSP-LR", y_te, p_ssp)]
    if kg_p is not None:
        panels.append(("Kaggle (external) - core SSP-LR", kg_y, kg_p))
    fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.2), squeeze=False)
    for ax, (title, y, p) in zip(axes[0], panels):
        xp, yo, n_ = cal_points(y, p)
        lim = max(xp.max(), yo.max()) * 1.1
        ax.plot([0, lim], [0, lim], color=GREY, ls=":", lw=1, label="Perfect calibration")
        ax.plot(xp, yo, color=BLUE, marker="o", ms=6, label="Observed (10 risk groups)")
        s, i = _calibration_slope_intercept(y, p)
        ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="Predicted risk", ylabel="Observed stroke rate",
               title=f"{title}\nslope {s:.2f}, intercept {i:+.2f}")
        ax.legend(fontsize=8.5, loc="upper left")
    save(fig, "Fig05_calibration", "Calibration on NHANES TEST and on Kaggle (external)")
except Exception as e:
    failed.append(("Fig05", repr(e)))

# ---------------------------------------------------------------- Fig 06 confusion matrices
try:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
    for ax, (title, pred) in zip(axes, [("Always 'no stroke'", np.zeros(len(y_te), int)),
                                        ("SSP-LR (Youden threshold)", (p_ssp >= thr_ssp).astype(int))]):
        cm = confusion_matrix(y_te, pred, labels=[0, 1])
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.6); ax.grid(False)
        for (r, c), v in np.ndenumerate(cm):
            ax.text(c, r, f"{v:,}", ha="center", va="center", fontsize=11, color=INK)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted no", "Predicted stroke"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["No stroke", "Stroke"])
        ax.set_title(title, loc="left")
    fig.suptitle("Confusion matrices on the locked TEST set", x=0.02, ha="left", fontsize=11)
    save(fig, "Fig06_confusion_matrices", "Baseline vs final model on TEST")
except Exception as e:
    failed.append(("Fig06", repr(e)))

# ---------------------------------------------------------------- Fig 07 external validation ROC
try:
    if kg_p is None:
        raise RuntimeError("Kaggle predictions not available")
    core_te_raw = core["model"].predict_proba(test[FEATURES_CORE])[:, 1]
    core_te = core["platt"].predict_proba(logit(np.clip(core_te_raw, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]
    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    for y, p, lab, col, ls in [(y_te, core_te, "NHANES TEST (internal)", BLUE, "-"),
                               (kg_y, kg_p, "Kaggle (external)", ORANGE, "--")]:
        f_, t_, _ = roc_curve(y, p)
        ax.plot(f_, t_, color=col, ls=ls, label=f"{lab}, AUC {roc_auc_score(y, p):.3f}")
    ax.plot([0, 1], [0, 1], color=GREY, ls=":", lw=1, label="Chance")
    ax.set(xlabel="1 - specificity", ylabel="Recall (sensitivity)",
           title="Core model (8 shared features)\ntrained on NHANES only")
    ax.legend(fontsize=8.5, loc="lower right")
    save(fig, "Fig07_external_validation_ROC", "Core model: NHANES TEST vs Kaggle")
except Exception as e:
    failed.append(("Fig07", repr(e)))

# ---------------------------------------------------------------- Fig 08 forest plot (survey ORs)
try:
    orx = pd.read_csv(OUT_DIR / "step08_survey_weighted_OR_exact.csv", index_col=0).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 8))
    ypos = np.arange(len(orx))
    sig = orx["p"] < 0.05
    ax.errorbar(orx["OR"], ypos, xerr=[orx["OR"] - orx["lo"], orx["hi"] - orx["OR"]], fmt="none",
                ecolor="#c9c7c0", lw=1.5)
    ax.scatter(orx["OR"], ypos, c=np.where(sig, BLUE, GREY), s=36, zorder=3)
    ax.axvline(1, color=MUTED, lw=1)
    ax.set_xscale("log"); ax.set_xticks([0.5, 0.75, 1, 1.5, 2, 3]); ax.set_xticklabels(["0.5", "0.75", "1", "1.5", "2", "3"])
    ax.set_yticks(ypos); ax.set_yticklabels(orx.index); ax.grid(axis="y", visible=False)
    ax.set_xlabel("Odds ratio (95% CI), log scale\nnumbers per 1 SD; yes/no and groups vs reference")
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=BLUE, label="p < 0.05"),
                       Line2D([], [], marker="o", ls="", color=GREY, label="not significant")],
              loc="lower right", fontsize=8.5)
    ax.set_title("Survey-weighted odds ratios (design-based CIs), NHANES TRAIN", loc="left")
    save(fig, "Fig08_forest_survey_OR", "Population-level risk factors")
except Exception as e:
    failed.append(("Fig08", repr(e)))

# ---------------------------------------------------------------- Fig 09 XAI agreement (ranks)
try:
    imp = pd.read_csv(OUT_DIR / "step08_xai_importance_exact.csv", index_col=0)
    imp = imp.sort_values("M3_shap_rank", ascending=False)
    fig, ax = plt.subplots(figsize=(6.6, 7))
    ypos = np.arange(len(imp))
    for col, lab, c, mk in [("M1_survey_wald_rank", "Survey-weighted OR", ORANGE, "s"),
                            ("M2_permutation_rank", "Permutation importance", AQUA, "^"),
                            ("M3_shap_rank", "SHAP", BLUE, "o")]:
        ax.scatter(imp[col], ypos, color=c, marker=mk, s=40, label=lab, zorder=3)
    ax.axvspan(0.5, 10.5, color="#eef4fc", zorder=0)
    ax.text(5.5, len(imp) - 0.3, "top 10", ha="center", color=MUTED, fontsize=8.5)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{f} *" if r else f for f, r in zip(imp.index, imp["robust"])])
    ax.set_xlim(0.5, 22.5); ax.invert_xaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("Importance rank (1 = most important)")
    ax.legend(loc="lower left", fontsize=8.5)
    ax.set_title("Do three explanation methods agree?  (* = robust: top 10 in all three)", loc="left", fontsize=10)
    save(fig, "Fig09_XAI_rank_agreement", "Rank of each risk factor under three XAI methods")
except Exception as e:
    failed.append(("Fig09", repr(e)))

# ---------------------------------------------------------------- Fig 10 XAI stability
try:
    st = pd.read_csv(OUT_DIR / "step08_xai_stability.csv", index_col=0).sort_values("robust_pct")
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    cols = [BLUE if v >= 80 else ("#86b6ef" if v >= 20 else "#d6d4cd") for v in st["robust_pct"]]
    ax.barh(st.index, st["robust_pct"], color=cols, height=0.65)
    ax.axvline(80, color=MUTED, ls="--", lw=1); ax.text(81, 0.2, "80% = stably robust", color=MUTED, fontsize=8.5)
    for i, v in enumerate(st["robust_pct"]):
        if v > 0:
            ax.text(v + 1, i, f"{v:.0f}%", va="center", fontsize=8.5, color=INK)
    ax.set_xlim(0, 110); ax.grid(axis="y", visible=False)
    ax.set_xlabel("% of 100 bootstrap resamples in the top 10 of all three methods")
    ax.set_title("Stability of the explanations", loc="left")
    save(fig, "Fig10_XAI_stability", "Bootstrap robustness of each risk factor")
except Exception as e:
    failed.append(("Fig10", repr(e)))

# ---------------------------------------------------------------- Fig 11 SSP-LR selection frequency
try:
    sf = pd.read_csv(OUT_DIR / "step07_SSPLR_selected_features.csv").sort_values("selection_freq")
    fig, ax = plt.subplots(figsize=(6.6, 10))
    ax.barh(sf["feature"], 100 * sf["selection_freq"], color=np.where(sf["kept"], BLUE, "#d6d4cd"), height=0.7)
    ax.axvline(60, color=MUTED, ls="--", lw=1); ax.text(61, 0.3, "threshold 60% (tuned)", color=MUTED, fontsize=8.5)
    ax.set_xlabel("% of 30 half-samples in which LASSO kept the column")
    ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8)
    ax.set_title("SSP-LR: stability selection (blue = kept)", loc="left")
    save(fig, "Fig11_SSPLR_selection_frequency", "Which model columns SSP-LR kept and how stably")
except Exception as e:
    failed.append(("Fig11", repr(e)))

# ---------------------------------------------------------------- Fig 12 sensitivity analyses
try:
    sv = pd.read_csv(OUT_DIR / "step10_sensitivity.csv", index_col=0)
    sv = sv[~sv.index.str.strip().str.startswith("(")].iloc[::-1]
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    ypos = np.arange(len(sv))
    ax.scatter(sv["roc_auc"], ypos, color=[BLUE if i.startswith("MAIN") else MUTED for i in sv.index], s=50, zorder=3)
    for yv_, v in zip(ypos, sv["roc_auc"]):
        ax.text(v + 0.002, yv_, f"{v:.3f}", va="center", fontsize=8.5, color=INK)
    ax.set_yticks(ypos); ax.set_yticklabels(sv.index); ax.grid(axis="y", visible=False)
    ax.set_xlim(0.80, 0.90); ax.set_xlabel("ROC-AUC on TEST")
    ax.set_title("Sensitivity analyses", loc="left")
    save(fig, "Fig12_sensitivity_analyses", "Main result vs S1-S4")
except Exception as e:
    failed.append(("Fig12", repr(e)))

print("DONE.\n", flush=True)
print("=" * 70)
print(f"STEP 11.1 SUMMARY - FIGURES saved in {FIG_DIR}")
print("=" * 70)
for name, cap in made:
    print(f"  OK      {name:<34} {cap}")
for name, err in failed:
    print(f"  FAILED  {name}: {err}")
print(f"\n{len(made)} figures made (each as .png and .pdf), {len(failed)} problems.")
print("=" * 70)

WORKING... creating figures in /Users/rafi_shariar/Documents/Thesis/ML- 9 Nov - NHANES/figures (about 1-2 min).
DONE.

STEP 11.1 SUMMARY - FIGURES saved in /Users/rafi_shariar/Documents/Thesis/ML- 9 Nov - NHANES/figures
  OK      Fig01_participant_flow             Inclusion flow and the stratified 80/20 split
  OK      Fig02_accuracy_paradox             Baseline vs final model: accuracy vs recall
  OK      Fig03_model_comparison             All models, nested CV, with statistical verdict
  OK      Fig04_ROC_PR_test                  ROC and PR curves of the final model and LR on TEST
  OK      Fig05_calibration                  Calibration on NHANES TEST and on Kaggle (external)
  OK      Fig06_confusion_matrices           Baseline vs final model on TEST
  OK      Fig07_external_validation_ROC      Core model: NHANES TEST vs Kaggle
  OK      Fig08_forest_survey_OR             Population-level risk factors
  OK      Fig09_XAI_rank_agreement           Rank of each risk factor under three 

In [26]:
# ============================================================
# STEP 11.2  FIX 3 FIGURES after visual check (overwrites the old files in figures/)
#   Fig04: label "Youden threshold" moved so it no longer crosses the chance line
#   Fig08: messy log-axis labels (6x10^-1, 4x10^0) removed; reference groups named
#   Fig09: legend moved below the plot so it no longer covers data points
# Needs: Cell 11.1 run in this session (re-uses its data and style).
# ============================================================
from matplotlib.ticker import NullFormatter, NullLocator
from matplotlib.lines import Line2D

for v in ["save", "p_ssp", "p_lr", "y_te", "thr_ssp"]:
    assert v in globals(), "Run Cell 11.1 first."
made, failed = [], []

# ---------------------------------------------------------------- Fig 04 (fixed)
try:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for p, lab, col, ls in [(p_ssp, "SSP-LR (final)", BLUE, "-"), (p_lr, "Logistic regression", ORANGE, "--")]:
        f_, t_, _ = roc_curve(y_te, p)
        axes[0].plot(f_, t_, color=col, ls=ls, label=f"{lab}, AUC {roc_auc_score(y_te, p):.3f}")
        pr_, rc_, _ = precision_recall_curve(y_te, p)
        axes[1].plot(rc_, pr_, color=col, ls=ls, label=f"{lab}, AP {average_precision_score(y_te, p):.3f}")
    axes[0].plot([0, 1], [0, 1], color=GREY, ls=":", lw=1, label="Chance")
    axes[1].axhline(prev, color=GREY, ls=":", lw=1, label=f"No-skill ({prev:.3f})")
    m_ = _metrics_at(y_te, p_ssp, thr_ssp)
    x0, y0 = 1 - m_["specificity"], m_["recall"]
    axes[0].scatter(x0, y0, color=BLUE, s=50, zorder=4, edgecolor="white")
    axes[0].annotate(f"Youden threshold (fixed on TRAIN)\nrecall {100*y0:.1f}%, specificity {100*m_['specificity']:.1f}%",
                     (x0, y0), xytext=(0.36, 0.63), fontsize=8.5, color=MUTED,
                     arrowprops=dict(arrowstyle="-", color=GREY))
    axes[0].set(xlabel="1 - specificity", ylabel="Recall (sensitivity)", title="ROC curve")
    axes[1].set(xlabel="Recall (sensitivity)", ylabel="Precision", title="Precision-recall curve")
    axes[0].legend(fontsize=8.5, loc="lower right"); axes[1].legend(fontsize=8.5, loc="upper right")
    fig.suptitle(f"Locked TEST set (n = {len(y_te):,}, {int(y_te.sum())} strokes)", x=0.02, ha="left", fontsize=11)
    save(fig, "Fig04_ROC_PR_test", "ROC and PR curves on TEST (label moved)")
except Exception as e:
    failed.append(("Fig04", repr(e)))

# ---------------------------------------------------------------- Fig 08 (fixed)
try:
    orx = pd.read_csv(OUT_DIR / "step08_survey_weighted_OR_exact.csv", index_col=0).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.4, 8.4))
    ypos = np.arange(len(orx)); sig = orx["p"] < 0.05
    ax.errorbar(orx["OR"], ypos, xerr=[orx["OR"] - orx["lo"], orx["hi"] - orx["OR"]], fmt="none",
                ecolor="#c9c7c0", lw=1.5)
    ax.scatter(orx["OR"], ypos, c=np.where(sig, BLUE, GREY), s=36, zorder=3)
    ax.axvline(1, color=MUTED, lw=1)
    ax.set_xscale("log")
    ax.xaxis.set_minor_locator(NullLocator()); ax.xaxis.set_minor_formatter(NullFormatter())
    ticks = [0.5, 0.75, 1, 1.5, 2, 3, 4]
    ax.set_xticks(ticks); ax.set_xticklabels([str(t) for t in ticks]); ax.set_xlim(0.42, 4.2)
    ax.set_yticks(ypos); ax.set_yticklabels(orx.index); ax.grid(axis="y", visible=False)
    ax.set_xlabel("Odds ratio (95% CI), log scale\n"
                  "Numbers: per 1 SD.  References: Non-Hispanic White, Some college/AA,\n"
                  "Married/partner, Never smoked (largest group of each).")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=BLUE, label="p < 0.05"),
                       Line2D([], [], marker="o", ls="", color=GREY, label="not significant")],
              loc="lower right", fontsize=8.5)
    ax.set_title("Survey-weighted odds ratios (design-based 95% CIs), NHANES TRAIN", loc="left", fontsize=10.5)
    save(fig, "Fig08_forest_survey_OR", "Forest plot (axis labels cleaned, references named)")
except Exception as e:
    failed.append(("Fig08", repr(e)))

# ---------------------------------------------------------------- Fig 09 (fixed)
try:
    imp = pd.read_csv(OUT_DIR / "step08_xai_importance_exact.csv", index_col=0).sort_values("M3_shap_rank", ascending=False)
    fig, ax = plt.subplots(figsize=(6.8, 7.6))
    ypos = np.arange(len(imp))
    for col, lab, c, mk in [("M1_survey_wald_rank", "Survey-weighted OR", ORANGE, "s"),
                            ("M2_permutation_rank", "Permutation importance", AQUA, "^"),
                            ("M3_shap_rank", "SHAP", BLUE, "o")]:
        ax.scatter(imp[col], ypos, color=c, marker=mk, s=40, label=lab, zorder=3, alpha=0.95)
    ax.axvspan(0.5, 10.5, color="#eef4fc", zorder=0)
    ax.text(5.5, len(imp) - 0.3, "top 10", ha="center", color=MUTED, fontsize=8.5)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{f} *" if r else f for f, r in zip(imp.index, imp["robust"])])
    ax.set_xlim(22.5, 0.5); ax.set_xticks([1, 5, 10, 15, 20]); ax.grid(axis="y", visible=False)
    ax.set_xlabel("Importance rank (1 = most important)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=3, fontsize=8.5)
    ax.set_title("Do three explanation methods agree?  (* = robust: top 10 in all three)", loc="left", fontsize=10)
    save(fig, "Fig09_XAI_rank_agreement", "XAI rank agreement (legend moved below)")
except Exception as e:
    failed.append(("Fig09", repr(e)))

print("=" * 66)
print("STEP 11.2 SUMMARY - FIGURE FIXES")
print("=" * 66)
for name, cap in made:
    print(f"  OK      {name:<30} {cap}")
for name, err in failed:
    print(f"  FAILED  {name}: {err}")
print("=" * 66)

STEP 11.2 SUMMARY - FIGURE FIXES
  OK      Fig04_ROC_PR_test              ROC and PR curves on TEST (label moved)
  OK      Fig08_forest_survey_OR         Forest plot (axis labels cleaned, references named)
  OK      Fig09_XAI_rank_agreement       XAI rank agreement (legend moved below)


In [27]:
# ============================================================
# STEP 11.3  THESIS TABLES -> one Excel file:  tables/thesis_tables.xlsx  (one sheet per table)
#
#  T01 Participant flow                  T06 XAI: agreement + stability
#  T02 Baseline characteristics          T07 External validation (Kaggle)
#  T03 Model comparison (nested CV)      T08 Risk-factor directions NHANES vs Kaggle
#  T04 Final results on TEST (95% CI)        (proper reference coding: never smoker = reference)
#  T05 Survey-weighted odds ratios       T09 Sensitivity analyses
#                                        T10 SSP-LR selected features
# Only reads saved results; T08 fits two small logistic regressions (descriptive).
# Needs: Cell 3.1 run in this session.
# ============================================================
import statsmodels.api as sm
from scipy import stats

TAB_DIR = ROOT / "tables"; TAB_DIR.mkdir(exist_ok=True)
print("WORKING... building tables (under 1 min).", flush=True)
T = {}
f3 = lambda v: f"{v:.3f}"
pct = lambda v: f"{100*v:.1f}%"

# ---- T01 flow
T["T01_flow"] = pd.read_csv(OUT_DIR / "step02_flow.csv").rename(columns={
    "step": "Step", "remaining": "Participants", "strokes": "Strokes", "removed": "Excluded"})

# ---- T02 baseline characteristics (whole analytic cohort)
coh = pd.read_csv(COHORT_CSV)
s1, s0 = coh[coh[TARGET] == 1], coh[coh[TARGET] == 0]
rows = [dict(Characteristic="N", Overall=f"{len(coh):,}", Stroke=f"{len(s1):,}", **{"No stroke": f"{len(s0):,}"},
             p="", Missing="")]
skewed = {"uacr", "glucose", "income_ratio"}
for c in NUMERIC_MAIN:
    miss = pct(coh[c].isna().mean())
    if c in skewed:
        fmt = lambda s: f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})"
        p = stats.mannwhitneyu(s1[c].dropna(), s0[c].dropna()).pvalue; lab = f"{c}, median (IQR)"
    else:
        fmt = lambda s: f"{s.mean():.1f} ({s.std():.1f})"
        p = stats.ttest_ind(s1[c].dropna(), s0[c].dropna(), equal_var=False).pvalue; lab = f"{c}, mean (SD)"
    rows.append(dict(Characteristic=lab, Overall=fmt(coh[c]), Stroke=fmt(s1[c]), **{"No stroke": fmt(s0[c])},
                     p=f"{p:.3g}", Missing=miss))
for c in BINARY_MAIN + CATEGORICAL_MAIN:
    levels = [1.0] if c in BINARY_MAIN else sorted(coh[c].dropna().unique())
    tab = pd.crosstab(coh[c], coh[TARGET])
    p = stats.chi2_contingency(tab)[1] if tab.shape[0] > 1 else np.nan
    for i, lv in enumerate(levels):
        cnt = lambda d: f"{int((d[c] == lv).sum()):,} ({100*(d[c] == lv).mean():.1f}%)"
        rows.append(dict(Characteristic=(f"{c}, yes" if c in BINARY_MAIN else f"{c}: {lv}"),
                         Overall=cnt(coh), Stroke=cnt(s1), **{"No stroke": cnt(s0)},
                         p=(f"{p:.3g}" if i == 0 else ""), Missing=(pct(coh[c].isna().mean()) if i == 0 else "")))
T["T02_baseline"] = pd.DataFrame(rows)

# ---- T03 model comparison
cmp_ = pd.read_csv(OUT_DIR / "step06_model_comparison.csv")
T["T03_model_comparison"] = pd.DataFrame({
    "Model": cmp_["model"], "PR-AUC mean (SD)": [f"{a:.3f} ({b:.3f})" for a, b in zip(cmp_.pr_auc, cmp_.pr_auc_sd)],
    "ROC-AUC": cmp_["roc_auc"].map(f3), "Recall": cmp_["recall"].map(pct), "Accuracy": cmp_["accuracy"].map(pct),
    "Holm-adjusted p vs best": cmp_["p_holm"].map(lambda v: f"{v:.3f}"), "Verdict": cmp_["verdict"],
    "Complexity rank": cmp_["complexity"]})

# ---- T04 TEST results
tr = pd.read_csv(OUT_DIR / "step07_test_results.csv")
keep = ["roc_auc", "pr_auc", "brier", "cal_slope", "cal_intercept", "youden_recall", "youden_accuracy",
        "youden_specificity", "youden_precision", "youden_npv", "youden_f1", "youden_f2", "f2_recall", "f2_accuracy"]
tr = tr[tr.metric.isin(keep)].copy()
is_pct = tr.metric.str.contains("recall|accuracy|specificity|precision|npv")
tr["cell"] = np.where(is_pct,
                      [f"{100*v:.1f}% ({100*l:.1f}-{100*h:.1f})" if pd.notna(l) else f"{100*v:.1f}%"
                       for v, l, h in zip(tr.value, tr.ci_low, tr.ci_high)],
                      [f"{v:.3f} ({l:.3f}-{h:.3f})" if pd.notna(l) else f"{v:.3f}"
                       for v, l, h in zip(tr.value, tr.ci_low, tr.ci_high)])
T["T04_test_results"] = tr.pivot(index="metric", columns="model", values="cell").reindex(keep).reset_index()

# ---- T05 survey ORs
orx = pd.read_csv(OUT_DIR / "step08_survey_weighted_OR_exact.csv", index_col=0)
T["T05_survey_OR"] = pd.DataFrame({"Factor": orx.index,
                                   "OR (95% CI)": [f"{o:.2f} ({l:.2f}-{h:.2f})" for o, l, h in zip(orx.OR, orx.lo, orx.hi)],
                                   "p": orx["p"].map(lambda v: "<0.001" if v < 0.001 else f"{v:.3f}")})

# ---- T06 XAI
imp = pd.read_csv(OUT_DIR / "step08_xai_importance_exact.csv", index_col=0)
stb = pd.read_csv(OUT_DIR / "step08_xai_stability.csv", index_col=0)
x = imp.join(stb[["robust_pct"]])
T["T06_XAI"] = pd.DataFrame({"Factor": x.index, "Rank survey-OR": x.M1_survey_wald_rank.astype(int),
                             "Rank permutation": x.M2_permutation_rank.astype(int), "Rank SHAP": x.M3_shap_rank.astype(int),
                             "Top-10 in all 3": np.where(x.robust, "yes", "no"),
                             "Bootstrap robustness %": x.robust_pct.round(0).astype(int)}
                            ).sort_values("Bootstrap robustness %", ascending=False)

# ---- T07 external validation
ev = pd.read_csv(OUT_DIR / "step09_external_validation.csv")
T["T07_external_validation"] = pd.DataFrame({
    "Model": ev.model, "Dataset": ev.dataset,
    "ROC-AUC (95% CI)": [f"{a:.3f} ({l:.3f}-{h:.3f})" for a, l, h in zip(ev.roc_auc, ev.roc_auc_lo, ev.roc_auc_hi)],
    "PR-AUC": ev.pr_auc.map(f3),
    "Recall (95% CI)": [f"{100*a:.1f}% ({100*l:.1f}-{100*h:.1f})" for a, l, h in zip(ev.recall, ev.recall_lo, ev.recall_hi)],
    "Accuracy": ev.accuracy.map(pct), "Calibration slope": ev.cal_slope.map(lambda v: f"{v:.2f}"),
    "Calibration intercept": ev.cal_intercept.map(lambda v: f"{v:+.2f}")})

# ---- T08 directions with proper reference coding
nh = pd.read_csv(TRAIN_CSV)
kg = pd.read_csv(ROOT / "data_external" / "kaggle_core_harmonised.csv")
sd = {c: nh[c].std() for c in ["age", "glucose", "bmi"]}
mu = {c: nh[c].mean() for c in ["age", "glucose", "bmi"]}


def design(d):
    d = d[FEATURES_CORE + [TARGET]].dropna()
    X = pd.DataFrame({f"{c} (per NHANES SD)": (d[c] - mu[c]) / sd[c] for c in ["age", "glucose", "bmi"]})
    for c in ["female", "hypertension", "heart_disease", "ever_married"]:
        X[c] = d[c].astype(float)
    X["smoking current vs never"] = (d["smoking"] == "current").astype(float)
    X["smoking former vs never"] = (d["smoking"] == "former").astype(float)
    return sm.add_constant(X), d[TARGET].values


def ors(d):
    X, y = design(d)
    r = sm.Logit(y, X).fit(disp=0)
    ci = np.exp(r.conf_int())
    return pd.DataFrame({"OR": np.exp(r.params), "lo": ci[0], "hi": ci[1], "p": r.pvalues}).drop(index="const"), len(y)


o_nh, n_nh = ors(nh)
o_kg, n_kg = ors(kg)
dirs = pd.DataFrame({
    "Factor": o_nh.index,
    f"NHANES TRAIN OR (95% CI), n={n_nh:,}": [f"{a:.2f} ({l:.2f}-{h:.2f})" for a, l, h in zip(o_nh.OR, o_nh.lo, o_nh.hi)],
    f"Kaggle OR (95% CI), n={n_kg:,}": [f"{a:.2f} ({l:.2f}-{h:.2f})" for a, l, h in zip(o_kg.OR, o_kg.lo, o_kg.hi)],
    "Same direction": np.where(np.sign(np.log(o_nh.OR)) == np.sign(np.log(o_kg.OR)), "yes", "NO"),
    "Significant in both (p<0.05)": np.where((o_nh.p < 0.05) & (o_kg.p < 0.05), "yes", "no")})
T["T08_directions"] = dirs

# ---- T09 sensitivity
sv = pd.read_csv(OUT_DIR / "step10_sensitivity.csv", index_col=0)
T["T09_sensitivity"] = pd.DataFrame({"Analysis": sv.index.str.strip(), "n": sv.n.astype(int), "Strokes": sv.strokes.astype(int),
                                     "ROC-AUC": sv.roc_auc.map(f3), "PR-AUC": sv.pr_auc.map(f3),
                                     "Recall": sv.recall.map(pct), "Accuracy": sv.accuracy.map(pct),
                                     "Calibration slope": sv.cal_slope.map(lambda v: f"{v:.2f}")})

# ---- T10 SSP-LR selected features
sf = pd.read_csv(OUT_DIR / "step07_SSPLR_selected_features.csv")
T["T10_SSPLR_features"] = pd.DataFrame({"Model column": sf.feature, "Selection frequency": (100 * sf.selection_freq).round(1),
                                        "Kept": np.where(sf.kept, "yes", "no"),
                                        "Odds ratio (final model)": sf.odds_ratio.map(lambda v: "" if pd.isna(v) else f"{v:.2f}")})

# ---- write Excel with readable column widths
xlsx = TAB_DIR / "thesis_tables.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    for name, df_ in T.items():
        df_.to_excel(w, sheet_name=name[:31], index=False)
        ws = w.sheets[name[:31]]
        for i, col in enumerate(df_.columns, start=1):
            width = max(len(str(col)), *(len(str(v)) for v in df_.iloc[:, i - 1].tolist())) + 2
            ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = min(width, 60)
print("DONE.\n", flush=True)

print("=" * 72)
print(f"STEP 11.3 SUMMARY - TABLES saved in {xlsx}")
print("=" * 72)
for name, df_ in T.items():
    print(f"  {name:<26} {len(df_):>3} rows")
print("\nT08 (new) - risk-factor directions with never-smoker as reference:")
for _, r in dirs.iterrows():
    print(f"  {r['Factor']:<28} NHANES {r.iloc[1]:<20} Kaggle {r.iloc[2]:<20} same direction: {r['Same direction']}")
print(f"  Same direction: {(dirs['Same direction'] == 'yes').sum()} of {len(dirs)}")
print("=" * 72)

WORKING... building tables (under 1 min).
DONE.

STEP 11.3 SUMMARY - TABLES saved in /Users/rafi_shariar/Documents/Thesis/ML- 9 Nov - NHANES/tables/thesis_tables.xlsx
  T01_flow                     5 rows
  T02_baseline                35 rows
  T03_model_comparison        16 rows
  T04_test_results            14 rows
  T05_survey_OR               30 rows
  T06_XAI                     22 rows
  T07_external_validation      6 rows
  T08_directions               9 rows
  T09_sensitivity              6 rows
  T10_SSPLR_features          52 rows

T08 (new) - risk-factor directions with never-smoker as reference:
  age (per NHANES SD)          NHANES 2.29 (2.06-2.54)     Kaggle 3.43 (2.72-4.34)     same direction: yes
  glucose (per NHANES SD)      NHANES 1.07 (1.02-1.14)     Kaggle 1.21 (1.08-1.35)     same direction: yes
  bmi (per NHANES SD)          NHANES 1.02 (0.95-1.09)     Kaggle 1.05 (0.88-1.25)     same direction: yes
  female                       NHANES 1.27 (1.10-1.47)     Kaggl